# 第 85 课：多控制器选主、去重工作队列与限速重试

**从一个声明式控制器，走向两个争夺同一控制权的控制者。**

上一课验证了低风险自动修复、危险漂移人工门槛、Status readiness 和本地 Lease；这一课加入持久工作队列、相同资源的事件合并、处理中的重复通知、指数重试、队列溢出与 Leader 切换。

本课仍使用真正的 **CPU PyTorch 因果 Transformer、Q/V LoRA、增量 KV 物理张量、双副本 WAL 和持久化配置**。不下载模型、不依赖上一课 `.py`、没有真实 Kubernetes 集群或网络。

> **重要范围：** 这里是**单进程串行模拟两个逻辑控制者**，不提供跨机器原子 CAS、etcd 共识、真正 Kubernetes leader election、RBAC/IAM 或毫秒级 RTO。租约、健康与队列延迟中的时间单位全部是**教学逻辑 tick**，不是秒或 GPU 延迟。

## 0. 学习目标与运行方法

建议按顺序阅读：先了解不丢事件的队列，再理解 Leader 票据；最后运行一次有实际 KV/WAL 的切主演练。

**完成标准：** 能解释为什么 watch 事件只是提示、为什么 `Get` 不应立刻删除任务、为什么 `Done/ack` 后可能仍需再入队，以及发生配置或容量不确定性时为何必须禁止放行新请求。

命令行脚本可直接运行：

```bash
python lesson_85_controller_workqueue_leader_election.py --checks-only
python lesson_85_controller_workqueue_leader_election.py --math-only
python lesson_85_controller_workqueue_leader_election.py
python lesson_85_controller_workqueue_leader_election.py --reload-run lesson_85_outputs/真实运行目录
```

Notebook 里的 `RUN_FULL_EXPERIMENT` 默认 `True`；练习的检查开关默认 `False`。从全新 Python 3 内核依次执行。输出建立新的时间戳目录，不覆盖旧实验，不保留可直接恢复整个运行时的 CPU KV 内存，而是通过 WAL 和模型权重重建。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第85课：多控制器选主、去重工作队列与限速重试。

完整离线、CPU PyTorch：真实自回归Transformer、Q/V LoRA与物理KV张量。
承接声明式调谐，演示双控制者租约、term围栏、持久工作队列、溢出阻断与接管后收敛。
仅为单进程模拟的双实例；不提供真实集群共识、远端存储、网络或IAM。
所有RTO/RPO单位是显式的教学逻辑tick；不宣称GPU毫秒性能。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要已有PyTorch环境；不会自动安装或下载。') from exc

VERSION = 'lesson85-workqueue-election-v1'
SOURCES = {
    'k8s_server_side_apply': 'https://kubernetes.io/docs/reference/using-api/server-side-apply/',
    'nist_config_management': 'https://csrc.nist.gov/pubs/sp/800/128/upd1/final',
    'sre_canary': 'https://sre.google/workbook/canarying-releases/',
    'sre_change': 'https://sre.google/sre-book/reliable-product-launches/',
    'torch_random': 'https://docs.pytorch.org/docs/stable/notes/randomness.html',
    'k8s_controller': 'https://kubernetes.io/docs/concepts/architecture/controller/',
    'workqueue': 'https://pkg.go.dev/k8s.io/client-go/util/workqueue',
    'leader_election': 'https://kubernetes.io/docs/concepts/cluster-administration/coordinated-leader-election/',
    'client_go_example': 'https://github.com/kubernetes/client-go/blob/master/examples/workqueue/main.go',
    'k8s_ssa': 'https://kubernetes.io/docs/reference/using-api/server-side-apply/',
    'operator_errors': 'https://www.redhat.com/en/blog/kubernetes-operators-best-practices',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(8501)
print('PyTorch:',torch.__version__,'CUDA可用:',torch.cuda.is_available())
print('本课运行范围: 单进程串行、真实CPU PyTorch/KV/WAL、无Kubernetes API')

PyTorch: 2.10.0+cpu CUDA可用: False
本课运行范围: 单进程串行、真实CPU PyTorch/KV/WAL、无Kubernetes API


## 1. 先复习上层依赖：token 边界与模型身份

为了证明控制系统影响真实推理，后半部分要在一个已训练结构的**固定教学权重**上执行 prefill、增量 decode、模型版本路由与 KV 一致性核对。这里不是对 Qwen 执行在线部署，也没有预训练权重。

该分词器、LoRA 和模型实现全部嵌入本 Notebook，绝不导入第84课或其他文件。模型参数与模板摘要是配置身份的一部分。内容 SHA 可发现意外修改，但不是对外部攻击者有效的数字签名。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

## 2. 真正的物理 KV 不是字典里的占位数字

本课程保留独立分配的 K/V Tensor：每层每头按实际位置写入、检查因果 attention、从 prefill 转入逐 token decode，验证分叉与终态回收。

推理缓存默认只能从持久 WAL 和原模型状态**重建**；没有把易失张量当作可跨机复用的状态。更早的课程实现了分页分配和终态释放，这里完整包含相关代码以便自运行。

In [4]:
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u


class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

## 3. 模型版本与序列身份都必须可重建

一台副本出现磁盘状态不一致，不应通过另一个副本的内存缓存假装恢复。模型权重、版本身份、WAL 事件与生成记录分别校验。下面保留两套可推理版本，随后再加入 Controller 的判断。

In [5]:
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

## 4. 持久化 WAL：先确认事件，再驱动真实推理

WAL 记录请求、逐 token 序号及终止状态；控制器修复不得重写用户请求历史。引入第二个 Leader 后，仍必须围栏旧集群 epoch 的写入，不能只检查 Python 对象里当前的 `holder` 字符串。

本课只处理本地同步双副本日志，不声称网络故障下获得真正的强一致性。

In [6]:
ZERO_HASH='0'*64

@dataclass
class LogicalRequest:
    rid: str
    tenant: str
    key: str
    version: str
    prompt_ids: list[int]
    max_new_tokens: int
    generated: list[int]=field(default_factory=list)
    terminal: str|None=None

    def identity(self):
        return {'tenant':self.tenant,'key':self.key,'version':self.version,
                'prompt_ids':self.prompt_ids,'max_new_tokens':self.max_new_tokens}


class Ledger:
    """WAL顺序应用的逻辑状态，不包含任何模型权重或物理KV。"""
    def __init__(self,codec:ByteMergeChatTokenizer,model_identity:dict):
        self.codec=codec
        self.model_identity=copy.deepcopy(model_identity)
        self.sessions:dict[str,LogicalRequest]={}
        self.seq=0;self.last_hash=ZERO_HASH;self.tick=0

    def to_json(self):
        return {'seq':self.seq,'last_hash':self.last_hash,'tick':self.tick,
                'sessions':{key:asdict(state) for key,state in sorted(self.sessions.items())}}

    def apply(self,record:dict):
        require(type(record) is dict and set(record)=={'seq','tick','kind','payload','prev','hash'},
                'WAL字段集合错误')
        require(type(record['seq']) is int and record['seq']==self.seq+1,'WAL序号必须连续')
        require(type(record['tick']) is int and record['tick']==self.tick+1,
                '本课tick是从1开始的每条持久事件序号')
        require(record['prev']==self.last_hash,'WAL前驱hash错误或从错误分叉接入')
        require(record['hash']==digest({k:v for k,v in record.items() if k!='hash'}),
                'WAL内容hash错误')
        kind=record['kind'];p=record['payload']
        require(type(p) is dict,'payload必须是dict')
        if kind=='create':
            require(set(p)=={'rid','tenant','key','version','prompt_ids','max_new_tokens'},
                    'create字段集合错误')
            rid=make_rid(p['tenant'],p['key'])
            require(rid==p['rid'] and rid not in self.sessions,'重复idempotency key/create冲突')
            require(p['version'] in self.model_identity['weights'],'WAL引用未知权重版本')
            require(type(p['prompt_ids']) is list and len(p['prompt_ids'])>0 and
                    len(p['prompt_ids'])<self.model_identity['max_length'], '提示token非法/超上下文')
            require(all(type(t) is int and 0<=t<self.codec.vocab_size for t in p['prompt_ids']),
                    'WAL提示token超词表')
            require(type(p['max_new_tokens']) is int and 1<=p['max_new_tokens']<=4 and
                    len(p['prompt_ids'])+p['max_new_tokens']<=self.model_identity['max_length'],
                    'WAL完成长度非法')
            self.sessions[rid]=LogicalRequest(**p)
        elif kind=='token':
            require(set(p)=={'rid','index','token','terminal'},'token字段集合错误')
            rid=p['rid'];require(rid in self.sessions,'追加token的请求不存在')
            session=self.sessions[rid]
            require(session.terminal is None,'终态后不可再追加token')
            require(type(p['index']) is int and p['index']==len(session.generated),
                    'token序号不连续')
            tok=p['token'];require(type(tok) is int and 0<=tok<self.codec.vocab_size,
                                  'token ID越界')
            require(p['terminal'] in {None,'eos','length'},'非法停止原因')
            next_length=len(session.generated)+1
            expected='eos' if tok==self.codec.eos_token_id else (
                     'length' if next_length==session.max_new_tokens else None)
            require(p['terminal']==expected,'EOS、长度终止不一致')
            session.generated.append(tok)
            session.terminal=p['terminal']
        else:
            raise LessonError('WAL事件类型非法')
        self.seq=record['seq'];self.tick=record['tick'];self.last_hash=record['hash']

    @classmethod
    def from_json(cls,codec,model_identity,data):
        require(type(data) is dict and set(data)=={'seq','last_hash','tick','sessions'},
                '快照状态字段错误')
        out=cls(codec,model_identity)
        require(type(data['seq']) is int and data['seq']>=0 and
                data['tick']==data['seq'] and type(data['last_hash']) is str and
                len(data['last_hash'])==64,'快照head错误')
        out.seq=data['seq'];out.tick=data['tick'];out.last_hash=data['last_hash']
        require(type(data['sessions']) is dict,'快照会话表必须为dict')
        for rid,s in data['sessions'].items():
            require(type(s) is dict and set(s)==set(LogicalRequest.__dataclass_fields__),
                    '快照会话字段非法')
            item=LogicalRequest(**s)
            require(item.rid==rid==make_rid(item.tenant,item.key),'快照id不一致')
            require(item.version in model_identity['weights'],'快照版本已失效')
            require(item.terminal in {None,'eos','length'},'快照停止状态非法')
            require(all(type(i) is int and 0<=i<codec.vocab_size for i in
                        item.prompt_ids+item.generated),'快照token越界')
            require(1<=item.max_new_tokens<=4 and
                    len(item.generated)<=item.max_new_tokens,'快照生成长度非法')
            if item.terminal=='eos':
                require(bool(item.generated) and item.generated[-1]==codec.eos_token_id,
                        '快照EOS状态不一致')
            if item.terminal=='length':
                require(len(item.generated)==item.max_new_tokens and
                        item.generated[-1]!=codec.eos_token_id,'快照长度终止不一致')
            if item.terminal is None:
                require(codec.eos_token_id not in item.generated and
                        len(item.generated)<item.max_new_tokens,'活动快照包含终态回答')
            out.sessions[rid]=item
        return out


def make_rid(tenant:str,key:str) -> str:
    for value,label in ((tenant,'租户'),(key,'幂等键')):
        require(type(value) is str and re.fullmatch('[A-Za-z0-9_-]{1,18}',value) is not None,
                label+'只能含1..18个ASCII字母数字_-')
    return tenant+'__'+key


def atomic_json(path:Path,payload:dict) -> None:
    """单机持久文件替换＋fsync；没有提供跨机器线性一致性。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    require(not tmp.exists(),'存在未清理的临时文件，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(payload,ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def make_record(ledger:Ledger,kind:str,payload:dict) -> dict:
    row={'seq':ledger.seq+1,'tick':ledger.tick+1,'kind':kind,
         'payload':copy.deepcopy(payload),'prev':ledger.last_hash}
    row['hash']=digest(row)
    trial=copy.deepcopy(ledger)
    trial.apply(row)
    return row


class DurableJournal:
    """严格顺序WAL：每行必须换行、hash chain，追加时flush+fsync。

    这是本地模拟“两个独立实例的磁盘”，不是分布式事务，不跨网复制fsync保证。
    故障后的未完成行默认拒绝；不静默截断或忽略。
    """
    def __init__(self,folder:Path,codec,model_identity,load_existing=False):
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.path=self.folder/'events.jsonl'
        self.snapshot_path=self.folder/'snapshot.json'
        self.codec=codec;self.model_identity=copy.deepcopy(model_identity)
        if not load_existing:
            require(not self.path.exists() and not self.snapshot_path.exists(),
                    '实例目录已有数据，禁止当作新副本覆盖')
            self.path.write_bytes(b'')
        else:
            require(self.path.is_file(),'恢复时WAL不存在')
        self.records=[];self.ledger=Ledger(codec,model_identity)
        if load_existing:self.recover()

    def read_records(self) -> list[dict]:
        raw=self.path.read_bytes()
        require(not raw or raw.endswith(b'\n'),'WAL末尾存在截断行，必须人工修复/显式恢复流程')
        rows=[];temp=Ledger(self.codec,self.model_identity)
        for n,line in enumerate(raw.splitlines(),1):
            require(line!=b'','WAL存在空行')
            try:
                row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError(f'WAL第{n}行不是有效JSON/UTF8') from exc
            temp.apply(row)
            rows.append(row)
        return rows

    def recover(self) -> dict:
        rows=self.read_records()  # 对全WAL做语法/hash链验证，快照不绕过完整性审计。
        all_state=Ledger(self.codec,self.model_identity)
        for row in rows:all_state.apply(row)
        snapshot_used=False;tail=rows
        if self.snapshot_path.exists():
            try:wrapped=json.loads(self.snapshot_path.read_text(encoding='utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('快照文件无法解码') from exc
            require(type(wrapped) is dict and set(wrapped)=={'payload','sha256'},'快照封装非法')
            data=wrapped['payload']
            require(wrapped['sha256']==digest(data),'快照内容摘要不匹配')
            require(type(data) is dict and set(data)=={'version','model_identity','state'},
                    '快照字段错误')
            require(data['version']==VERSION and data['model_identity']==self.model_identity,
                    '快照模型/分词身份不匹配')
            restored=Ledger.from_json(self.codec,self.model_identity,data['state'])
            require(restored.seq<=len(rows),'快照seq超出WAL实际长度')
            prefix=Ledger(self.codec,self.model_identity)
            for row in rows[:restored.seq]:prefix.apply(row)
            require(prefix.to_json()==restored.to_json(),
                    '快照与WAL前缀的内容不一致：不允许信任重算hash后的假快照')
            tail=rows[restored.seq:]
            for row in tail:restored.apply(row)
            require(restored.to_json()==all_state.to_json(),'快照加尾WAL不等于全量恢复')
            self.ledger=restored;snapshot_used=True
        else:self.ledger=all_state
        self.records=rows
        return {'wal_records_verified':len(rows),'snapshot_used':snapshot_used,
                'tail_events_semantically_applied':len(tail),'head_hash':self.ledger.last_hash}

    def append_row(self,row:dict) -> None:
        # 在触盘前先检查下一状态，避免错误事件落盘后再被拒绝。
        probe=copy.deepcopy(self.ledger);probe.apply(row)
        with self.path.open('ab') as f:
            payload=(canonical(row)+'\n').encode('utf-8')
            f.write(payload);f.flush();os.fsync(f.fileno())
        self.ledger=probe
        self.records.append(copy.deepcopy(row))

    def checkpoint(self) -> dict:
        data={'version':VERSION,'model_identity':copy.deepcopy(self.model_identity),
              'state':self.ledger.to_json()}
        atomic_json(self.snapshot_path,{'payload':data,'sha256':digest(data)})
        return {'seq':self.ledger.seq,'last_hash':self.ledger.last_hash,
                'size_bytes':self.snapshot_path.stat().st_size,
                'contents':'logical token IDs and request states; NOT raw KV tensors'}

## 5. 独立副本与恢复：为什么旧请求仍绑定原模型？

会话创建时固定版本；之后的控制器选主、重试、灰度配置或磁盘恢复，不能替换该请求已经承诺的模型版本。恢复时重放 WAL 与 token 前缀，再用增量 KV 与全上下文 logits 对照。

In [7]:
def artifact_identity(tok,models:dict[str,TinyChatLM]) -> dict:
    require(type(models) is dict and set(models)=={'stable','canary'},'模型版本集合不完整')
    configs=[asdict(m.cfg) for m in models.values()]
    require(configs[0]==configs[1],'两个版本配置不同')
    return {'tokenizer':digest(tok.contract()),'weights':{k:state_hash(m.state_dict())
              for k,m in sorted(models.items())},'config':configs[0],
              'max_length':configs[0]['max_length']}


class Replica:
    def __init__(self,name:str,folder:Path,tok,models,capacity_blocks=64,
                 block_size=4,load_existing=False):
        require(name in {'A','B'},'副本名必须A/B')
        self.name=name;self.tok=tok;self.models=copy.deepcopy(models)
        self.identity=artifact_identity(tok,self.models)
        self.folder=Path(folder)
        self.journal=DurableJournal(self.folder,tok,self.identity,load_existing)
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(capacity_blocks,block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(tok,self.models,self.pool)
        self.up=True
        self.last_recovery={'wal_records_verified':0,'snapshot_used':False,
                            'tail_events_semantically_applied':0}
        if load_existing:self.recover_cache()

    @property
    def ledger(self):return self.journal.ledger

    def _expected(self,rid):
        require(rid in self.pool.sessions and rid in self.runner.last_logits,
                '活动请求没有对应真实KV；拒绝偷偷从空上下文解码')
        logits=self.runner.last_logits[rid]
        require(bool(torch.isfinite(logits).all()),'KV输出异常')
        return int(logits.argmax())

    def prevalidate(self,row:dict) -> None:
        require(self.up,'实例已宕机')
        require(artifact_identity(self.tok,self.models)==self.identity,
                '运行模型权重漂移，拒绝生成/应用新事件')
        test=copy.deepcopy(self.ledger);test.apply(row)
        kind=row['kind'];payload=row['payload']
        if kind=='token':
            rid=payload['rid']
            require(rid in self.pool.sessions and rid in self.runner.last_logits,
                    '副本缺失活动KV：先恢复再继续')
            require(self._expected(rid)==payload['token'],
                    '复制到另一个副本时增量输出不一致：权重/前缀/位置存在漂移')

    def accept(self,row:dict) -> None:
        self.prevalidate(row)
        self.journal.append_row(row)
        p=row['payload']
        try:
            if row['kind']=='create':
                self.runner.open_prefill(p['rid'],p['version'],p['prompt_ids'])
            else:
                if p['terminal'] is None:self.runner.step(p['rid'],p['token'])
                else:self.runner.close(p['rid'])
            self.pool.audit()
        except Exception:
            # 已持久事件不能撒谎称“未发生”；该副本必须标记不可用并离线重建。
            self.up=False
            raise

    def create(self,payload:dict) -> dict:
        record=make_record(self.ledger,'create',payload)
        self.accept(record)
        return record

    def token(self,payload:dict) -> dict:
        record=make_record(self.ledger,'token',payload)
        self.accept(record)
        return record

    def recover_cache(self) -> dict:
        require(artifact_identity(self.tok,self.models)==self.identity,
                '恢复实例权重身份不匹配')
        details=self.journal.recover()
        # 丢弃所有易失KV，依据逻辑会话从最早提示重建。
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(self.pool.capacity_blocks,self.pool.block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(self.tok,self.models,self.pool)
        total=0;active=0
        for rid,state in sorted(self.ledger.sessions.items()):
            if state.terminal is not None:continue
            active+=1
            self.runner.open_prefill(rid,state.version,state.prompt_ids)
            total+=len(state.prompt_ids)
            for t in state.generated:
                require(self._expected(rid)==t,
                        '恢复后同一前缀的贪心输出不同，拒绝身份/模型漂移')
                self.runner.step(rid,t)
                total+=1
            self.runner.assert_consistent(rid)
        self.up=True
        details.update({'active_sessions_rebuilt':active,'tokens_recomputed':total,
                        'kv_tensors_restored_from_disk':False,'kv_live_blocks':self.pool.usage()['live_blocks']})
        self.last_recovery=details
        return details

    def audit(self):
        active={k for k,v in self.ledger.sessions.items() if v.terminal is None}
        require(set(self.pool.sessions)==active and set(self.runner.last_logits)==active,
                '逻辑活跃会话与KV实体不一致')
        max_diff=0.0
        for rid in sorted(active):
            st=self.ledger.sessions[rid]
            require(self.pool.sessions[rid].tokens==st.prompt_ids+st.generated,
                    'KV与已确认的逻辑token历史不一致')
            require(self.pool.sessions[rid].version==st.version,'KV与持久版本绑定不一致')
            max_diff=max(max_diff,self.runner.assert_consistent(rid))
        u=self.pool.audit()
        return {'replica':self.name,'logical_seq':self.ledger.seq,'logical_requests':len(self.ledger.sessions),
                'active_sessions':len(active),'kv_live_blocks':u['live_blocks'],
                'max_cache_full_logits_diff':max_diff,'models':self.identity['weights']}


def _request_payload(tok,tenant,key,version,text,max_new_tokens):
    rid=make_rid(tenant,key)
    require(version in {'stable','canary'},'模型版本非法')
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=4,'输出预算非法')
    prompt=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                   add_generation_prompt=True)
    return {'rid':rid,'tenant':tenant,'key':key,'version':version,
            'prompt_ids':prompt,'max_new_tokens':max_new_tokens}

## 6. 单点控制 epoch 仍是另一个围栏

控制器 Leader term 与推理主节点 epoch **不是同一个计数**：Leader term 标识哪个控制者允许发起调谐；cluster epoch 标识哪个推理副本允许继续提交 WAL token。

生产集群需要可靠的共享权威和线性化更新。这里通过一个串行本地控制点与磁盘原子替换模拟逻辑，不模拟同一瞬间两个真实进程并发争抢磁盘。

In [8]:
class SimulatedCrash(RuntimeError):
    """明确模拟写盘之后、客户端收到ACK之前的中断，不执行真实进程杀死。"""


def save_model_artifacts(path:Path,tok,models) -> dict:
    require(not Path(path).exists(),'模型artifact不可覆盖')
    identity=artifact_identity(tok,models)
    blob={'version':VERSION,'identity':identity,'tokenizer':tok.contract(),
          'weights':{version:{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                     for version,model in models.items()}}
    torch.save(blob,path)
    return identity


def load_model_artifacts(path:Path):
    # 仅加载本课自己生成、可信来源的纯张量与dict；不执行外部网络下载。
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'artifact版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer/chat template身份不匹配')
    identity=blob['identity']
    cfg=ToyConfig(**identity['config'])
    models={}
    for name,values in blob['weights'].items():
        model=inject_lora(TinyChatLM(cfg),rank=4,alpha=8.0)
        model.load_state_dict(values,strict=True)
        models[name]=model.eval().requires_grad_(False)
    require(artifact_identity(tok,models)==identity,'模型artifact哈希不匹配')
    return tok,models


class Coordinator:
    """一个教学用串行控制器，持久化epoch与ACK审计；不是分布式共识。"""
    def __init__(self,root:Path,tok,models,policy='sync',seed=8101,
                 capacity_blocks=64,load_existing=False):
        require(policy in {'sync','async'},'确认模式只能sync/async')
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.control_path=self.root/'control.json'
        self.artifact_path=self.root/'models.pt'
        self.tok=tok;self.models=models
        self.model_identity=artifact_identity(tok,models)
        self.capacity_blocks=capacity_blocks
        self.seed=seed
        if load_existing:
            self.control=self._read_control()
            require(self.control['model_identity']==self.model_identity,
                    '重载模型与原先已确认事件的权重/词表身份不符')
            require(self.control['policy']==policy,'控制器确认策略不可暗中更改')
        else:
            require(not self.control_path.exists() and not self.artifact_path.exists(),
                    '根目录已包含集群持久状态，禁止覆盖')
            save_model_artifacts(self.artifact_path,tok,models)
            self.control={'version':VERSION,'active':'A','epoch':1,'policy':policy,
                          'model_identity':self.model_identity,'ack_seq':0,'ack_tick':0,
                          'ack_history':[],'failovers':[],'quarantined':[],'seed':seed,
                          'capacity_blocks':capacity_blocks}
            atomic_json(self.control_path,self.control)
        self.rep={name:Replica(name,self.root/name,tok,models,capacity_blocks,
                               load_existing=load_existing) for name in ('A','B')}
        self.updates_acknowledged=0
        for name in self.control['quarantined']:
            require(name!=self.control['active'],'活跃主不能被隔离')
            self.rep[name].shutdown()
        self._check_artifacts()

    def _read_control(self):
        try:state=json.loads(self.control_path.read_text(encoding='utf-8'))
        except (UnicodeDecodeError,json.JSONDecodeError,FileNotFoundError) as exc:
            raise LessonError('控制指针无法读取') from exc
        require(type(state) is dict and
                set(state)=={'version','active','epoch','policy','model_identity','ack_seq',
                             'ack_tick','ack_history','failovers','quarantined','seed','capacity_blocks'},
                '控制记录字段非法')
        require(state['version']==VERSION and state['active'] in {'A','B'} and
                type(state['epoch']) is int and state['epoch']>=1,'控制代际非法')
        require(type(state['ack_seq']) is int and state['ack_seq']>=0 and
                state['ack_tick']==state['ack_seq'],'ACK水位非法')
        require(type(state['quarantined']) is list and
                set(state['quarantined'])<={'A','B'} and
                state['active'] not in state['quarantined'],
                '隔离副本列表非法')
        return state

    def _check_artifacts(self):
        token,models=load_model_artifacts(self.artifact_path)
        require(artifact_identity(token,models)==self.model_identity,
                '模型artifact与当前控制身份不符')
        for r in self.rep.values():
            require(r.identity==self.model_identity,'副本模型身份漂移')
            require(artifact_identity(r.tok,r.models)==r.identity,'运行中模型权重被改写')

    def _fence(self,epoch:int):
        durable=self._read_control()
        require(type(epoch) is int and epoch==durable['epoch'] and
                durable['active']==self.control['active'] and
                durable['epoch']==self.control['epoch'],
                'epoch过期或控制指针已被其他控制器切换；拒绝陈旧主节点写入')
        require(self.rep[durable['active']].up,'活跃副本不可用')
        return self.rep[durable['active']]

    def _standby(self):return self.rep['B' if self.control['active']=='A' else 'A']

    def replicate_pending(self):
        primary=self.rep[self.control['active']];target=self._standby()
        require(target.name not in self.control['quarantined'],
                '备机日志已隔离，必须显式丢弃旧分叉并重建')
        require(target.up,'备机不可用，不能确认同步复制')
        require(target.identity==primary.identity,'复制副本模型身份不相等')
        left=target.journal.records;right=primary.journal.records
        require(len(left)<=len(right) and
                all(a==b for a,b in zip(left,right)),
                '日志发生分叉：不可自动拼接，必须隔离并人工处理')
        count=0
        for row in right[len(left):]:
            target.accept(row);count+=1
        require(target.ledger.last_hash==primary.journal.records[target.ledger.seq-1]['hash']
                if target.ledger.seq else target.ledger.last_hash==ZERO_HASH,
                '复制完成后head hash不一致')
        return count

    def _ack(self,row:dict):
        require(type(row) is dict and 'seq' in row and 'hash' in row,'ACK没有WAL证据')
        primary=self.rep[self.control['active']]
        require(primary.ledger.seq>=row['seq'] and
                primary.journal.records[row['seq']-1]==row,'主机日志和ACK不匹配')
        already_acked=row['seq']<=self.control['ack_seq']
        if already_acked:
            require(any(e['seq']==row['seq'] and e['hash']==row['hash']
                        for e in self.control['ack_history']),
                    '历史ACK证据与当前日志不匹配')
        elif self.control['policy']=='sync':
            standby=self._standby()
            require(standby.name not in self.control['quarantined'] and
                    standby.up and standby.ledger.seq>=row['seq'] and
                    standby.journal.records[row['seq']-1]==row,'同步ACK前备机没有持久化该事件')
        if row['seq']>self.control['ack_seq']:
            require(row['seq']==self.control['ack_seq']+1,'不能越过未确认的先前事件')
            self.control['ack_seq']=row['seq'];self.control['ack_tick']=row['tick']
            self.control['ack_history'].append({'seq':row['seq'],'hash':row['hash'],
                                               'epoch':self.control['epoch']})
            atomic_json(self.control_path,self.control)
            self.updates_acknowledged+=1
        return {'seq':row['seq'],'hash':row['hash'],'epoch':self.control['epoch'],
                'durability':'primary+standby' if self.control['policy']=='sync' else 'primary_only'}

    def _commit(self,row:dict,crash_after_durable=False):
        primary=self.rep[self.control['active']]
        primary.accept(row)   # 先写盘fsync，再响应或复制。
        if self.control['policy']=='sync':self.replicate_pending()
        if crash_after_durable:
            raise SimulatedCrash('WAL已fsync但控制ACK未写入；客户端必须同键对账重试')
        return self._ack(row)

    def start(self,epoch,tenant,key,version='stable',text='hello',max_new_tokens=4,
              crash_after_durable=False):
        active=self._fence(epoch)
        payload=_request_payload(self.tok,tenant,key,version,text,max_new_tokens)
        rid=payload['rid']
        if rid in active.ledger.sessions:
            item=active.ledger.sessions[rid]
            require(item.identity()=={k:payload[k] for k in item.identity()},
                    '同一幂等键与先前请求内容/版本不一致')
            row=next(r for r in active.journal.records if r['kind']=='create'
                     and r['payload']['rid']==rid)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'new_event':False,'ack':ack}
        row=make_record(active.ledger,'create',payload)
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'new_event':True,'ack':ack}

    def step(self,epoch,rid,expected_index:int,crash_after_durable=False):
        active=self._fence(epoch)
        require(rid in active.ledger.sessions,'请求不存在')
        item=active.ledger.sessions[rid]
        require(type(expected_index) is int and expected_index>=0,'续传序号非法')
        if expected_index<len(item.generated):
            row=next(r for r in active.journal.records if r['kind']=='token' and
                     r['payload']['rid']==rid and r['payload']['index']==expected_index)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'index':expected_index,'token':row['payload']['token'],
                    'terminal':row['payload']['terminal'],'new_event':False,'ack':ack}
        require(expected_index==len(item.generated),'不能跳过尚未产生的token')
        require(item.terminal is None,'请求已经终止')
        tok=active._expected(rid)
        stop='eos' if tok==self.tok.eos_token_id else (
             'length' if expected_index+1==item.max_new_tokens else None)
        row=make_record(active.ledger,'token',{'rid':rid,'index':expected_index,
                                              'token':tok,'terminal':stop})
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'index':expected_index,'token':tok,'terminal':stop,
                'new_event':True,'ack':ack}

    def checkpoint(self):
        return {name:self.rep[name].journal.checkpoint() for name in ('A','B')}

    def promote(self,epoch,target='B',allow_data_loss=False,rpo_budget_ticks=0,
                rto_budget_ticks=12,tokens_per_tick=32,wal_records_per_tick=16):
        require(type(allow_data_loss) is bool,'数据丢失授权必须显式bool')
        require(target in self.rep and target!=self.control['active'],'接管目标非法')
        persisted=self._read_control()
        require(type(epoch) is int and epoch==self.control['epoch'] and
                persisted==self.control, '接管用epoch过期或控制状态发生变化')
        # 故障后原主机可down；只有这个教学控制指针继续提供围栏，不由原主授权。
        require(all(type(v) is int and v>=0 for v in (rpo_budget_ticks,rto_budget_ticks))
                and tokens_per_tick>0 and wal_records_per_tick>0,'恢复目标/预算非法')
        old_active=self.control['active'];standby=self.rep[target]
        require(standby.up,'备机不可用')
        self._check_artifacts()
        # 在新对象中只从本地持久日志及快照恢复，易失KV不复制过来。
        fresh=Replica(target,self.root/target,self.tok,self.models,self.capacity_blocks,
                      load_existing=True)
        require(artifact_identity(fresh.tok,fresh.models)==self.model_identity,
                '灾难恢复模型/模板指纹不一致')
        audit=fresh.audit()
        ack_seq=self.control['ack_seq']
        lost=max(0,ack_seq-fresh.ledger.seq)
        require(fresh.ledger.seq<=self.rep[old_active].ledger.seq,
                '备机不能凭空比主机多出旧epoch事件')
        # 完整一致的确认前缀不能只是seq相等；还要比较哈希链前缀。
        primary_records=self.rep[old_active].journal.records
        require(fresh.journal.records==primary_records[:len(fresh.journal.records)],
                '备机历史并非旧主已提交前缀；禁止接管分叉日志')
        rpo_gap=max(0,self.control['ack_tick']-fresh.ledger.tick)
        rto=2+math.ceil(fresh.last_recovery['wal_records_verified']/wal_records_per_tick)+(
             math.ceil(fresh.last_recovery['tokens_recomputed']/tokens_per_tick))
        if (lost>0 or rpo_gap>rpo_budget_ticks) and not allow_data_loss:
            raise LessonError('未复制的已ACK事件存在，自动接管必须阻断并报告RPO风险')
        if rto>rto_budget_ticks:
            raise LessonError('预计恢复逻辑tick超出RTO门槛；本课拒绝自动接管')
        report={'from':old_active,'to':target,'old_epoch':epoch,'new_epoch':epoch+1,
                'sync_mode':self.control['policy'],'ack_seq_before':ack_seq,
                'target_seq':fresh.ledger.seq,'lost_acked_events':lost,
                'rpo_gap_logical_ticks':rpo_gap,'rpo_objective_ticks':rpo_budget_ticks,
                'rto_simulated_ticks':rto,'rto_objective_ticks':rto_budget_ticks,
                'rto_formula':'2 + ceil(WAL_verified / records_per_tick) + ceil(KV_replay_tokens / tokens_per_tick)',
                'recovered_active_sessions':audit['active_sessions'],
                'recovered_tokens_recomputed':fresh.last_recovery['tokens_recomputed'],
                'data_loss_override':allow_data_loss and lost>0,
                'no_real_time_measurement':True}
        new_control=copy.deepcopy(self.control)
        new_control['active']=target;new_control['epoch']=epoch+1
        new_control['ack_seq']=fresh.ledger.seq;new_control['ack_tick']=fresh.ledger.tick
        if lost>0:
            new_control['ack_history']=[row for row in new_control['ack_history']
                                        if row['seq']<=fresh.ledger.seq]
            new_control['quarantined']=sorted(set(new_control['quarantined'])|{old_active})
        new_control['failovers'].append(copy.deepcopy(report))
        # 本地控制指针fsync后再关闭旧主；失效旧epoch依赖这个唯一可信控制点。
        atomic_json(self.control_path,new_control)
        self.rep[old_active].shutdown()
        standby.shutdown()
        self.rep[target]=fresh
        self.control=new_control
        return report

    def audit(self):
        require(self._read_control()==self.control,'控制内存与持久状态不一致')
        self._check_artifacts()
        active=self.rep[self.control['active']]
        require(active.up,'当前活跃副本不可用')
        return {'active':self.control['active'],'epoch':self.control['epoch'],
                'ack_seq':self.control['ack_seq'], 'artifacts':self.model_identity,
                'active_audit':active.audit(),
                'standby_head':self._standby().ledger.seq}


def _replica_shutdown(self):
    for sid in list(self.pool.sessions):
        if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
        self.runner.close(sid)
    require(self.pool.audit()['live_blocks']==0,'关闭副本仍有物理KV泄漏')
    self.up=False

Replica.shutdown=_replica_shutdown


def restore_controller(root:Path) -> Coordinator:
    root=Path(root)
    tok,models=load_model_artifacts(root/'models.pt')
    control=json.loads((root/'control.json').read_text(encoding='utf-8'))
    result=Coordinator(root,tok,models,policy=control['policy'],seed=control['seed'],
                       capacity_blocks=control['capacity_blocks'],load_existing=True)
    result.audit()
    return result

## 7. 容量、配置与最小权限

本课继承上一课经过显式审计的唯一自动修复字段：`telemetry_sample_percent`。

不会自动改写路由权重、允许租户、模型身份、WAL 模式、物理 KV 保留量或准入开关。Kubernetes Controller 的理想形态是**不断比较期望和实际**，而不是按事件携带的旧内容盲写。([Kubernetes Controller 概念](https://kubernetes.io/docs/concepts/architecture/controller/))

`watch` 事件只是通知某个资源可能变化；真正处理时要重新读取最新磁盘状态，因此队列里只放有界的资源键与版本指纹，不存原始 prompt、token、秘密或可直接执行的“修改指令”。

In [9]:
def capacity_projection(replica:Replica,available_blocks:int,reserve_blocks:int,
                        planned_new_session_lengths:list[int]|None=None):
    """每会话保守按未来最长上下文估算页数；不计共享前缀收益，避免把优化当容量保证。"""
    require(type(available_blocks) is int and available_blocks>0 and
            type(reserve_blocks) is int and reserve_blocks>=0,'容量必须非负整数且池大小为正')
    new=[] if planned_new_session_lengths is None else planned_new_session_lengths
    require(type(new) is list and all(type(n) is int and 0<n<=replica.pool.cfg.max_length for n in new),
            '新请求长度预算必须为正且不超过上下文')
    block=replica.pool.block_size
    current=0;worst=0;active=0
    for st in replica.ledger.sessions.values():
        if st.terminal is not None:continue
        active+=1
        length=len(st.prompt_ids)+len(st.generated)
        remaining=st.max_new_tokens-len(st.generated)
        require(0<=remaining<=4 and length+remaining<=replica.pool.cfg.max_length,
                '会话剩余token预算不合法')
        current+=math.ceil(length/block)
        worst+=math.ceil((length+remaining)/block)
    worst+=sum(math.ceil(x/block) for x in new)
    total=worst+reserve_blocks
    bytes_per_block=replica.pool.bytes_per_block
    return {'active_sessions':active,'current_required_blocks':current,
            'projected_session_blocks':worst,'emergency_reserved_blocks':reserve_blocks,
            'total_required_blocks':total,'available_blocks':available_blocks,
            'headroom_blocks':available_blocks-total,'allowed':available_blocks>=total,
            'bytes_per_block':bytes_per_block,'projected_reserved_bytes':total*bytes_per_block,
            'physical_capacity_bytes':available_blocks*bytes_per_block,
            'model':'per-session conservative; no prefix share credit; live KV only; excludes model weights'}

In [10]:
@dataclass(frozen=True)
class ServingConfig:
    schema: str
    model_identity_hash: str
    tokenizer_template_hash: str
    wal_policy: str
    route_canary_percent: int
    routing_salt: str
    allowed_tenants: tuple[str, ...]
    max_active_sessions: int
    reserve_blocks: int
    admission_enabled: bool
    telemetry_sample_percent: int

    def validate(self):
        require(self.schema == 'serving-config-v1', '未知配置模式，不能忽略字段')
        for label, value in (('模型身份', self.model_identity_hash), ('分词模板', self.tokenizer_template_hash)):
            require(type(value) is str and bool(re.fullmatch('[0-9a-f]{64}', value)), label+'哈希非法')
        require(self.wal_policy == 'sync', '本课零RPO发布只支持同步持久化ACK')
        require(type(self.route_canary_percent) is int and 0 <= self.route_canary_percent <= 100,
                'canary比例必须为整数0..100')
        require(type(self.telemetry_sample_percent) is int and 0 <= self.telemetry_sample_percent <= 100,
                '观测抽样比例必须为整数0..100')
        require(type(self.routing_salt) is str and bool(re.fullmatch('[A-Za-z0-9_-]{1,24}', self.routing_salt)),
                '路由salt非法')
        require(type(self.allowed_tenants) is tuple and 0 < len(self.allowed_tenants) <= 8 and
                all(type(t) is str and bool(re.fullmatch('[a-z][a-z0-9_-]{0,15}',t)) for t in self.allowed_tenants) and
                len(set(self.allowed_tenants)) == len(self.allowed_tenants) and
                tuple(sorted(self.allowed_tenants)) == self.allowed_tenants,
                '租户列表必须排序、唯一且受限')
        require(type(self.max_active_sessions) is int and 1 <= self.max_active_sessions <= 16,
                '会话上限非法')
        require(type(self.reserve_blocks) is int and 2 <= self.reserve_blocks <= 128,
                '物理KV应保留至少两页紧急余量')
        require(type(self.admission_enabled) is bool, '准入开关必须是显式bool')

    def wire(self) -> dict:
        self.validate()
        data=asdict(self);data['allowed_tenants']=list(self.allowed_tenants)
        return data

    @staticmethod
    def from_wire(data:dict):
        require(type(data) is dict and set(data)==set(ServingConfig.__dataclass_fields__),
                '配置字段不完整/有未知字段，不进行隐式降级')
        require(type(data['allowed_tenants']) is list, 'JSON中的租户必须是列表')
        obj=ServingConfig(**{**data,'allowed_tenants':tuple(data['allowed_tenants'])})
        obj.validate()
        require(obj.wire()==data,'配置无法无损往返')
        return obj


def initial_serving_config(cluster:Coordinator) -> ServingConfig:
    identity=artifact_identity(cluster.tok,cluster.models)
    result=ServingConfig('serving-config-v1',digest(identity),identity['tokenizer'],'sync',
                         0,'course83',('alpha','beta'),8,4,True,10)
    result.validate();return result


def changed_config(cfg:ServingConfig, **changes) -> ServingConfig:
    data=cfg.wire();data.update(changes)
    result=ServingConfig.from_wire(data)
    return result


def config_diff(before:ServingConfig,after:ServingConfig) -> list[dict]:
    a,b=before.wire(),after.wire()
    return [{'path':key,'old':a[key],'new':b[key],
             'severity':'critical' if key in {'model_identity_hash','tokenizer_template_hash',
               'wal_policy','allowed_tenants','max_active_sessions','reserve_blocks','admission_enabled'}
               else 'controlled'} for key in sorted(a) if a[key]!=b[key]]

In [11]:
class ChangeLedger:
    """本地hash链+单独封存head；不能证明外部主体身份或防恶意重写两者。"""
    def __init__(self,path:Path,load_existing=False):
        self.path=Path(path);self.head_path=self.path.with_suffix('.head.json')
        if load_existing:
            self.verify();return
        require(not self.path.exists() and not self.head_path.exists(),'不能覆盖历史变更审计')
        self.path.parent.mkdir(parents=True,exist_ok=True)
        self.path.touch()
        atomic_json(self.head_path,{'count':0,'head':ZERO_HASH})

    def verify(self) -> dict:
        try: lines=self.path.read_bytes().splitlines();head=json.loads(self.head_path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('变更审计缺失/损坏') from exc
        require(type(head) is dict and set(head)=={'count','head'},'审计head字段非法')
        last=ZERO_HASH;records=[]
        for i,raw in enumerate(lines,1):
            try:record=json.loads(raw.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('审计JSON行损坏') from exc
            require(type(record) is dict and set(record)=={'seq','tick','event','data','prev','hash'},
                    '变更审计字段不匹配')
            require(type(record['seq']) is int and record['seq']==i and record['prev']==last,
                    '审计事件序号/前向链错误')
            require(type(record['tick']) is int and 0<=record['tick']<=100000,
                    '逻辑tick非法')
            require(type(record['event']) is str and bool(record['event']) and
                    type(record['data']) is dict,'审计事件非法')
            core={k:v for k,v in record.items() if k!='hash'}
            require(record['hash']==digest(core),'审计内容被改写')
            last=record['hash'];records.append(record)
        require(head=={'count':len(records),'head':last},'审计内容与封存head不一致')
        return {'count':len(records),'head':last,'events':[r['event'] for r in records]}

    def add(self,tick:int,event:str,data:dict) -> dict:
        previous=self.verify()
        require(type(tick) is int and 0<=tick<=100000,'tick非法')
        require(type(event) is str and bool(re.fullmatch('[a-z_]{2,48}',event)),'事件名非法')
        require(type(data) is dict and len(canonical(data).encode())<=2048,
                '审计字段过大：不允许写入原始prompt、token和密钥')
        require(not any('prompt' in k.lower() or 'secret' in k.lower() or 'token_ids' in k.lower()
                        for k in data),'原始prompt/token不能写入审计')
        last=self.path.read_bytes().splitlines()
        if last:
            previous_record=json.loads(last[-1]);require(tick>=previous_record['tick'],'审计tick不能倒退')
        core={'seq':previous['count']+1,'tick':tick,'event':event,
              'data':copy.deepcopy(data),'prev':previous['head']}
        row={**core,'hash':digest(core)}
        with self.path.open('ab') as f:
            f.write((canonical(row)+'\n').encode('utf-8'));f.flush();os.fsync(f.fileno())
        atomic_json(self.head_path,{'count':row['seq'],'head':row['hash']})
        self.verify();return row


class ConfigStore:
    """单进程串行CAS实验，不能替代K8s apiserver/etcd/租约或真正分布式共识。"""
    def __init__(self,root:Path,cluster:Coordinator,load_existing=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.cluster=cluster
        self.ptr=self.root/'active.json';self.versions=self.root/'versions'
        self.applied=self.root/'replicas';self.audit=ChangeLedger(self.root/'changes.jsonl',load_existing)
        self.versions.mkdir(exist_ok=True)
        for name in ('A','B'):(self.applied/name).mkdir(parents=True,exist_ok=True)
        if load_existing:
            self.read_active(require_healthy=False)
        else:
            require(not self.ptr.exists(),'活动指针不可覆盖')
            initial=initial_serving_config(cluster)
            self._stage(1,initial)
            atomic_json(self.ptr,self._pointer(1,initial))
            self.audit.add(0,'initial_release',{'generation':1,'config_sha':digest(initial.wire())})
            self.read_active(require_healthy=True)

    @staticmethod
    def _pointer(generation:int, config:ServingConfig) -> dict:
        return {'generation':generation,'sha':digest(config.wire())}

    def _version_path(self,generation:int) -> Path:
        require(type(generation) is int and generation>=1,'配置代际非法')
        return self.versions/f'{generation:06d}.json'

    def _replica_path(self,name:str,generation:int) -> Path:
        require(name in {'A','B'},'副本非法')
        return self.applied/name/f'{generation:06d}.json'

    def read_pointer(self) -> dict:
        try:state=json.loads(self.ptr.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('活动版本指针不可读取') from exc
        require(type(state) is dict and set(state)=={'generation','sha'},'活动指针字段不对')
        require(type(state['generation']) is int and state['generation']>=1 and
                type(state['sha']) is str and bool(re.fullmatch('[0-9a-f]{64}',state['sha'])),
                '活动指针值非法')
        return state

    def _load_config(self,path:Path) -> ServingConfig:
        try:content=json.loads(path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('副本或版本配置损坏') from exc
        return ServingConfig.from_wire(content)

    def read_active(self,require_healthy=True) -> tuple[dict,ServingConfig]:
        p=self.read_pointer();spec=self._load_config(self._version_path(p['generation']))
        require(digest(spec.wire())==p['sha'],'活动指针与被选版本哈希不一致')
        if require_healthy:self.assert_aligned(p,spec)
        return p,spec

    def assert_aligned(self,p:dict|None=None,spec:ServingConfig|None=None) -> dict:
        if p is None or spec is None:p,spec=self.read_active(False)
        observed={}
        for name in ('A','B'):
            got=self._load_config(self._replica_path(name,p['generation']))
            require(got.wire()==spec.wire(),f'{name}副本配置漂移；停止新请求与自动接管')
            observed[name]=digest(got.wire())
        model_identity=artifact_identity(self.cluster.tok,self.cluster.models)
        require(spec.model_identity_hash==digest(model_identity) and
                spec.tokenizer_template_hash==model_identity['tokenizer'],
                '运行模型或模板身份与配置不一致')
        require(self.cluster.control['policy']==spec.wal_policy,'WAL确认模式发生漂移')
        require(self.cluster._read_control()==self.cluster.control,'控制指针与内存不一致')
        require(self.cluster.model_identity==model_identity,'实际模型权重身份变化')
        require(self.cluster.capacity_blocks>=spec.reserve_blocks,'物理KV页容量不足')
        return {'generation':p['generation'],'sha':p['sha'],'replicas':observed,
                'epoch':self.cluster.control['epoch'],'model_identity_hash':digest(model_identity)}

    def _immutable_json(self,path:Path,content:dict):
        if path.exists():
            existing=self._load_config(path)
            require(existing.wire()==content,'预置文件已存在但内容不同：不能自动覆盖')
        else:
            atomic_json(path,content)

    def _stage(self,generation:int,cfg:ServingConfig,fail_after:str|None=None):
        data=cfg.wire()
        self._immutable_json(self._version_path(generation),data)
        for name in ('A','B'):
            self._immutable_json(self._replica_path(name,generation),data)
            if fail_after=='stage_'+name:
                raise SimulatedCrash('只预置了部分副本：活动指针未改变，不能宣称发布成功')
        # 预置阶段的文件只能通过指针切换变为生效配置。
        for name in ('A','B'):
            require(self._load_config(self._replica_path(name,generation)).wire()==data,
                    '预置副本内容不匹配')

    def snapshot(self) -> dict:
        p,s=self.read_active(require_healthy=True)
        control=self.cluster.control
        return {'pointer':copy.deepcopy(p),'config_sha':digest(s.wire()),
                'epoch':control['epoch'],'active':control['active'],
                'ack_seq':control['ack_seq'],'control_sha':digest(control),
                'A_seq':self.cluster.rep['A'].ledger.seq,
                'B_seq':self.cluster.rep['B'].ledger.seq,
                'artifact_sha':digest(self.cluster.model_identity)}

    def probe_kv(self) -> dict:
        self.assert_aligned()
        reports={};max_delta=0.0
        tok=self.cluster.tok
        prompt=tok.apply_chat_template([{'role':'user','content':'hello'}],
                                       tokenize=True,add_generation_prompt=True)
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            require(replica.up,'副本未运行，发布预检拒绝')
            require(artifact_identity(tok,replica.models)==self.cluster.model_identity,
                    '副本实际模型权重身份与发布不匹配')
            for version in ('stable','canary'):
                pool=PagedKVPool(32,4,next(iter(replica.models.values())).cfg,set(replica.models))
                runner=CachedModelRunner(tok,replica.models,pool)
                sid='preflight_'+name+'_'+version
                runner.open_prefill(sid,version,prompt)
                delta=runner.assert_consistent(sid)
                max_delta=max(max_delta,delta)
                runner.close(sid)
                require(pool.audit()['live_blocks']==0,'preflight的物理KV没有释放')
                reports[name+'_'+version]={'max_cache_full_diff':delta,'released':True}
        return {'max_cache_full_diff':max_delta,'checked':reports,'real_tensor_kv':True}

    def safe_capacity(self,proposed:ServingConfig,planned_new_session_lengths:list[int]|None=None) -> dict:
        proposed.validate()
        result={}
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            check=capacity_projection(replica,self.cluster.capacity_blocks,proposed.reserve_blocks,
                                      planned_new_session_lengths)
            require(check['allowed'],f'{name}可恢复KV容量不够；不准上线')
            require(check['active_sessions']<=proposed.max_active_sessions,
                    f'{name}现有会话数超过新配置上限')
            result[name]=check
        return result

    def preflight(self,candidate:ServingConfig,tick:int,ttl_ticks:int=4) -> dict:
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '使用非负的教学逻辑tick；TTL为1..20')
        candidate.validate();p,old=self.read_active(require_healthy=True)
        require(candidate.model_identity_hash==old.model_identity_hash and
                candidate.tokenizer_template_hash==old.tokenizer_template_hash and
                candidate.wal_policy==old.wal_policy,
                '模型/模板/WAL身份变更需要独立的新artifact迁移，不能只修改配置字段')
        for name in ('A','B'):
            for st in self.cluster.rep[name].ledger.sessions.values():
                require(st.terminal is not None or st.tenant in candidate.allowed_tenants,
                        '存在活跃会话时不能撤销该租户访问权限')
        checks=self.safe_capacity(candidate)
        probes=self.probe_kv()
        snap=self.snapshot()
        report={'kind':'release_preflight','base':snap,'candidate':candidate.wire(),
                'candidate_sha':digest(candidate.wire()),'issued_tick':tick,'expires_tick':tick+ttl_ticks,
                'capacity':checks,'probe':probes,'diff':config_diff(old,candidate),
                'decision':'allow','limitations':'logical_ticks_single_controller_no_authenticated_approval'}
        report['seal']=digest(report)
        return report

    def verify_preflight(self,proof:dict,candidate:ServingConfig,tick:int) -> None:
        require(type(proof) is dict and 'seal' in proof and
                proof['seal']==digest({k:v for k,v in proof.items() if k!='seal'}),
                '发布证据完整性错误')
        require(proof['kind']=='release_preflight' and proof['decision']=='allow',
                '证据没有明确放行')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '预检证据尚未生效或已超过逻辑TTL')
        require(proof['candidate']==candidate.wire() and proof['candidate_sha']==digest(candidate.wire()),
                '预检证据绑定的是另一个候选')
        require(proof['base']==self.snapshot(),
                '预检后控制epoch/ACK/副本状态/生效配置改变，必须重新取证')
        self.safe_capacity(candidate)
        self.probe_kv()

    @staticmethod
    def approval(candidate:ServingConfig,base_generation:int,actor='local_operator') -> dict:
        return {'actor':actor,'ticket':'CHG-0083','approved':True,
                'candidate_sha':digest(candidate.wire()),'expected_generation':base_generation}

    def _approval(self,approve:dict,candidate:ServingConfig,expected_generation:int):
        require(type(approve) is dict and set(approve)==
                {'actor','ticket','approved','candidate_sha','expected_generation'},
                '审批记录字段不匹配')
        require(approve['approved'] is True and approve['expected_generation']==expected_generation and
                approve['candidate_sha']==digest(candidate.wire()) and
                type(approve['actor']) is str and bool(approve['actor']) and
                approve['ticket']=='CHG-0083','模拟审批没有绑定这次变更或代际')

    def publish(self,candidate:ServingConfig,proof:dict,approval:dict,
                expected_generation:int,tick:int,fail_after:str|None=None) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(type(expected_generation) is int and p['generation']==expected_generation,
                '配置CAS预期代际过期；拒绝并行覆盖')
        self._approval(approval,candidate,expected_generation)
        self.verify_preflight(proof,candidate,tick)
        generation=expected_generation+1
        self._stage(generation,candidate,fail_after)
        # 提交前再次检查；这只是单进程串行CAS模型，跨进程仍需真正的控制服务。
        require(self.read_pointer()==p,'预置期间活动指针变化，停止提交')
        self.verify_preflight(proof,candidate,tick)
        atomic_json(self.ptr,self._pointer(generation,candidate))
        if fail_after=='after_commit':
            raise SimulatedCrash('活动指针已切换但客户端未收到ACK：必须读取实际版本，不允许自动再次发布')
        self.assert_aligned()
        self.audit.add(tick,'release_committed',{'generation':generation,'candidate_sha':digest(candidate.wire()),
                                                'from_generation':expected_generation})
        return {'generation':generation,'active_sha':digest(candidate.wire()),
                'diff':proof['diff'],'ack':'committed','cas_expected':expected_generation}

    def detect_drift(self) -> dict:
        p,s=self.read_active(False)
        findings=[]
        for name in ('A','B'):
            try:observed=self._load_config(self._replica_path(name,p['generation']))
            except LessonError as exc:
                findings.append({'replica':name,'status':'invalid','reason':str(exc)})
                continue
            for change in config_diff(s,observed):
                findings.append({'replica':name,'status':'drift',**change})
        actual=artifact_identity(self.cluster.tok,self.cluster.models)
        if digest(actual)!=s.model_identity_hash:
            findings.append({'replica':'models','status':'drift','path':'model_identity_hash'})
        if self.cluster.control['policy']!=s.wal_policy:
            findings.append({'replica':'wal','status':'drift','path':'wal_policy'})
        return {'generation':p['generation'],'findings':findings,'healthy':not findings}

    def reconcile(self,name:str,approval:dict,tick:int,expected_generation:int):
        require(name in {'A','B'},'副本名非法')
        p,s=self.read_active(False)
        require(p['generation']==expected_generation,'修复预期代际过期')
        self._approval(approval,s,expected_generation)
        self._load_config(self._version_path(p['generation']))
        # 单控制者的显式修复动作，不掩盖WAL/真实权重漂移，不自动删除历史。
        path=self._replica_path(name,p['generation'])
        before=self.detect_drift()['findings']
        require(any(item['replica']==name for item in before),'未检测到该副本漂移，拒绝无意义覆盖')
        atomic_json(path,s.wire())
        self.assert_aligned()
        self.audit.add(tick,'drift_reconciled',{'generation':expected_generation,
                                                'replica':name,'restored_sha':digest(s.wire())})
        return {'reconciled':name,'generation':expected_generation,'healthy':True}

    def _has_commit_receipt(self,generation:int,sha:str) -> bool:
        self.audit.verify()
        for line in self.audit.path.read_text(encoding='utf-8').splitlines():
            record=json.loads(line)
            data=record['data']
            if record['event'] in {'initial_release','release_committed','release_reconciled'} and \
               data.get('generation')==generation and data.get('candidate_sha',data.get('config_sha'))==sha:
                return True
        return False

    def resolve_uncertain_commit(self,candidate:ServingConfig,proof:dict,approval:dict,
                                 expected_generation:int,tick:int) -> dict:
        """指针已经提交而ACK/审计未完成：读取真实指针，不盲目再递增版本。"""
        self._approval(approval,candidate,expected_generation)
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '无法核对原始预检证据')
        require(proof.get('candidate_sha')==digest(candidate.wire()) and
                proof.get('base',{}).get('pointer',{}).get('generation')==expected_generation,
                '含糊ACK绑定其他候选或前驱代际')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '含糊ACK过久，需要重新建立人工处置流程')
        p,s=self.read_active(require_healthy=True)
        require(p==self._pointer(expected_generation+1,candidate) and s.wire()==candidate.wire(),
                '未找到与原提交完全一致的活动指针；不能假装已经发布')
        require(self.cluster.control['epoch']==proof['base']['epoch'] and
                self.cluster.control['ack_seq']==proof['base']['ack_seq'] and
                self.cluster.control['active']==proof['base']['active'],
                '提交后出现新流量或切主，不能直接补写旧审计')
        recorded=self._has_commit_receipt(p['generation'],p['sha'])
        if not recorded:
            self.audit.add(tick,'release_reconciled',{'generation':p['generation'],
                              'candidate_sha':p['sha'],'from_generation':expected_generation})
        return {'generation':p['generation'],'committed':True,'audit_reconciled':not recorded,
                'pointer_advanced_again':False}

    def route(self,tenant:str,key:str,text:str='hello',max_new_tokens:int=4) -> dict:
        p,s=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置尚未有完整提交审计，先核对不确定ACK')
        require(s.admission_enabled,'当前配置停止接受新请求')
        require(tenant in s.allowed_tenants,'租户不在授权列表')
        active=self.cluster.rep[self.cluster.control['active']]
        require(active.up,'活动实例不可用')
        rid=make_rid(tenant,key)
        if rid in active.ledger.sessions:
            version=active.ledger.sessions[rid].version
        else:
            current=sum(st.terminal is None for st in active.ledger.sessions.values())
            require(current<s.max_active_sessions,'达到活动会话上限')
            future_prompt=self.cluster.tok.apply_chat_template(
                [{'role':'user','content':text}],tokenize=True,add_generation_prompt=True)
            self.safe_capacity(s,[len(future_prompt)+max_new_tokens])
            bucket=int(digest({'tenant':tenant,'key':key,'salt':s.routing_salt})[:12],16)%100
            version='canary' if bucket<s.route_canary_percent else 'stable'
        receipt=self.cluster.start(self.cluster.control['epoch'],tenant,key,version,text,max_new_tokens)
        return {**receipt,'version':version,'config_generation':p['generation'],
                'config_sha':p['sha'],'pinned_to_wal':True}

    def step(self,rid:str) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置缺少提交审计，停止生成')
        active=self.cluster.rep[self.cluster.control['active']]
        require(rid in active.ledger.sessions,'会话不存在')
        st=active.ledger.sessions[rid]
        require(st.terminal is None,'会话已经结束')
        return self.cluster.step(self.cluster.control['epoch'],rid,len(st.generated))

## 8. 恢复安全门槛与声明式自动修复

自动修复受控字段之前，检查真实两副本配置、模型哈希、WAL 控制代际和 KV 容量。危险漂移会转到 `ManualReview`，并将 Ready 设为 False。

本课程的人工审批是**本地显式教学夹具**，不是生产工单身份认证。

In [12]:
class RecoveryGate:
    def __init__(self,store:ConfigStore):self.store=store

    def preflight(self,target:str,tick:int,ttl_ticks:int=4) -> dict:
        store=self.store;cluster=store.cluster
        require(target in {'A','B'} and target!=cluster.control['active'],'目标应是另一个副本')
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '逻辑时间/TTL非法')
        p,s=store.read_active(require_healthy=True)
        require(s.wal_policy=='sync','本课端到端零RPO验收要求sync')
        standby=cluster.rep[target]
        require(standby.up,'接管目标不可用')
        require(standby.ledger.seq>=cluster.control['ack_seq'],
                '目标没有全部已ACK日志，不能无损接管')
        require(standby.journal.records==cluster.rep[cluster.control['active']].journal.records[:len(standby.journal.records)],
                '接管目标日志不是当前主的前缀')
        capacity=store.safe_capacity(s)[target]
        audit=standby.audit();require(audit['max_cache_full_logits_diff']<=2e-5,
                                      '真实KV与全上下文输出不一致')
        payload={'kind':'recovery_preflight','generation':p['generation'],
                 'config_sha':p['sha'],'epoch':cluster.control['epoch'],
                 'old_active':cluster.control['active'],'target':target,
                 'ack_seq':cluster.control['ack_seq'],'target_seq':standby.ledger.seq,
                 'target_head':standby.ledger.last_hash,'control_sha':digest(cluster.control),
                 'issued_tick':tick,'expires_tick':tick+ttl_ticks,'capacity':capacity,
                 'target_kv_max_abs_diff':audit['max_cache_full_logits_diff'],
                 'target_active_sessions':audit['active_sessions'],
                 'estimated_rto_logical_ticks':2+math.ceil(len(standby.journal.records)/16)+
                    math.ceil(sum(len(st.prompt_ids)+len(st.generated) for st in standby.ledger.sessions.values()
                                  if st.terminal is None)/32),
                 'decision':'allow'}
        payload['seal']=digest(payload)
        return payload

    def promote(self,proof:dict,tick:int,rto_budget_ticks:int=40) -> dict:
        store=self.store;cluster=store.cluster
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '恢复门槛证据被修改')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '恢复证据已过期')
        require(proof['decision']=='allow','恢复证据未放行')
        p,s=store.read_active(require_healthy=True)
        require(p['generation']==proof['generation'] and p['sha']==proof['config_sha'],
                '恢复前配置代际被改动')
        require(cluster.control['epoch']==proof['epoch'] and
                cluster.control['active']==proof['old_active'] and
                cluster.control['ack_seq']==proof['ack_seq'] and
                digest(cluster.control)==proof['control_sha'],
                '恢复前控制epoch/ACK发生变化，必须重新取证')
        target=proof['target']
        require(cluster.rep[target].ledger.seq==proof['target_seq'] and
                cluster.rep[target].ledger.last_hash==proof['target_head'],
                '恢复目标日志被改动')
        store.safe_capacity(s)
        require(type(rto_budget_ticks) is int and rto_budget_ticks>=proof['estimated_rto_logical_ticks'],
                '恢复逻辑RTO预算过小；不允许先关闭健康主机再失败')
        # 明确模拟主机宕机，但并不接触真实主机或外部网络。
        cluster.rep[cluster.control['active']].shutdown()
        result=cluster.promote(proof['epoch'],target,rpo_budget_ticks=0,
                               rto_budget_ticks=rto_budget_ticks)
        require(result['lost_acked_events']==0 and result['rpo_gap_logical_ticks']==0,
                '切主结果不满足零RPO')
        require(cluster.control['epoch']==proof['epoch']+1 and
                cluster.control['active']==target,'旧主没有被epoch隔离')
        store.assert_aligned()
        audit=cluster.rep[target].audit()
        require(audit['max_cache_full_logits_diff']<=2e-5,'重新计算的KV与全前向不一致')
        check={'failover':result,'active':target,'epoch':cluster.control['epoch'],
               'old_epoch_fenced':True,'actual_kv_rebuilt':True,
               'max_kv_full_logits_diff':audit['max_cache_full_logits_diff'],
               'active_sessions_rebuilt':audit['active_sessions'],
               'config_generation_preserved':store.read_pointer()['generation']==proof['generation'],
               'zero_rpo_verified':True,'logical_time_only':True}
        store.audit.add(tick,'recovery_accepted',{'generation':proof['generation'],
                                                'epoch':cluster.control['epoch'],'ack_seq':cluster.control['ack_seq']})
        return check

In [13]:
@dataclass(frozen=True)
class ReconcilePolicy:
    auto_paths: tuple[str,...] = ('telemetry_sample_percent',)
    grace_ticks: int = 2
    retry_base_ticks: int = 2
    retry_cap_ticks: int = 8
    max_retries: int = 3
    status_fresh_ticks: int = 2

    def validate(self):
        # 课程明确授权的唯一低风险字段；即使调用者改配置，也不能扩大自动更改权限。
        require(type(self.auto_paths) is tuple and self.auto_paths == ('telemetry_sample_percent',),
                '本课自动修复权限仅限telemetry_sample_percent，不能把路由/租户/容量加入')
        for name in ('grace_ticks','retry_base_ticks','retry_cap_ticks','max_retries','status_fresh_ticks'):
            v=getattr(self,name)
            require(type(v) is int and v>=1, name+'必须为正整数')
        require(self.retry_cap_ticks>=self.retry_base_ticks,'最大退避不能小于基础退避')


def classify_drift(findings:list[dict],policy:ReconcilePolicy) -> dict:
    """分类时严格白名单，未知/损坏/跨副本混合一律人工处理。"""
    policy.validate()
    require(type(findings) is list,'漂移需要列表')
    if not findings:
        return {'mode':'healthy','replica':None,'paths':[]}
    paths=[]
    for finding in findings:
        require(type(finding) is dict,'非法漂移记录')
        paths.append(f"{finding.get('replica','?')}.{finding.get('path',finding.get('status','?'))}")
    if len(findings)==1:
        item=findings[0]
        if (item.get('status')=='drift' and item.get('replica') in ('A','B')
                and item.get('path') in policy.auto_paths
                and item.get('severity')=='controlled'):
            return {'mode':'auto_safe','replica':item['replica'],'paths':paths}
    return {'mode':'manual','replica':None,'paths':sorted(paths)}

## 9. 从单一 Lease 到多个控制者

Kubernetes 的 Leader Election 借助共享 Lease 对控制平面组件进行协调；Leader 需在期限内续约，故障后其他候选可以争取租约。([官方协调选主说明](https://kubernetes.io/docs/concepts/cluster-administration/coordinated-leader-election/))

本课保留 `holder/term/cluster_epoch/issued_tick/expires_tick/token`。任何调谐写操作都校验当前票据，并要求**存储的票据与持有的票据完全相同**。即使 holder 和 term 未变，旧续约 token 也不再有效。

**租约失效不等于旧实例物理死亡。** 围栏应在受保护操作入口再次核验，否则进程暂停后恢复时可能以过期身份写入。

In [14]:
class LocalLease:
    """本地教学CAS/租约围栏；不是跨进程互斥、鉴权或K8s leader election。"""
    def __init__(self,path:Path,cluster:Coordinator):
        self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
        self.cluster=cluster

    def read(self):
        if not self.path.exists():return None
        try:raw=json.loads(self.path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('本地lease损坏，停止调谐') from exc
        require(type(raw) is dict and set(raw)=={'holder','term','cluster_epoch','issued_tick',
                    'expires_tick','version','token','seal'},'lease字段非法')
        require(raw['version']==VERSION and raw['seal']==digest({k:v for k,v in raw.items() if k!='seal'}),
                'lease完整性错误')
        require(type(raw['holder']) is str and bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',raw['holder'])),
                'lease持有者非法')
        require(all(type(raw[k]) is int and raw[k]>=0 for k in ('term','cluster_epoch','issued_tick','expires_tick')),
                'lease计数非法')
        require(raw['term']>=1 and raw['cluster_epoch']>=1 and raw['expires_tick']>=raw['issued_tick'],
                'lease时间非法')
        return raw

    def acquire(self,holder:str,tick:int,ttl:int=20) -> dict:
        bounded_str(holder,24,'holder')
        require(bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',holder)), 'holder字符非法')
        require(type(tick) is int and tick>=0 and type(ttl) is int and 2<=ttl<=100,
                '教学租约tick/ttl非法')
        current_epoch=self.cluster._read_control()['epoch']
        old=self.read()
        if old and old['expires_tick']>=tick and old['cluster_epoch']==current_epoch:
            require(old['holder']==holder,'当前lease由另一控制者持有；旧控制者不得越权')
            return copy.deepcopy(old)
        term=old['term']+1 if old else 1
        core={'holder':holder,'term':term,'cluster_epoch':current_epoch,
              'issued_tick':tick,'expires_tick':tick+ttl,'version':VERSION,
              'token':digest({'holder':holder,'term':term,'epoch':current_epoch,'issued':tick})}
        value={**core,'seal':digest(core)}
        atomic_json(self.path,value)
        return value

    def verify(self,lease:dict,tick:int):
        require(type(tick) is int and tick>=0,'tick非法')
        current=self.read()
        require(current is not None and type(lease) is dict and lease==current,
                '控制者lease过期/不属于当前控制者')
        require(current['issued_tick']<=tick<=current['expires_tick'],
                '控制者lease超过有效tick')
        require(current['cluster_epoch']==self.cluster._read_control()['epoch']
                and current['cluster_epoch']==self.cluster.control['epoch'],
                '控制epoch发生切换，必须重新选举/取lease')
        return True

## 10. 控制调谐仍是 level-based，不是 event replay

相同逻辑 tick 重入不重复写审计；低风险漂移先经历宽限，暂时失败执行有限指数退避；权限或模板改变必须人工处理。

注意：`ready`、`observed_generation` 和 `desired_sha` 都是**读过真实配置之后**才更新的状态，不能直接把通知事件上的代际当成已经观察成功的事实。

In [15]:
class ReconcileLoop:
    """Level-based单步调谐。函数每次都读取磁盘真实期望与观测，而非执行排队命令。

    本地状态/指纹不是身份签名，控制者只模拟单进程串行；严禁与真实集群控制混淆。
    """
    STATUS_FIELDS={'version','observed_generation','desired_sha','cluster_epoch','ready','live',
                   'reason','mode','finding_sha','first_seen_tick','retry_count','next_retry_tick',
                   'last_tick','last_success_tick','last_error','quarantined','paths'}

    def __init__(self,root:Path,store:ConfigStore,policy:ReconcilePolicy|None=None,
                 load_existing=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.store=store;self.policy=policy or ReconcilePolicy();self.policy.validate()
        self.status_path=self.root/'status.json'
        self.lease=LocalLease(self.root/'lease.json',store.cluster)
        self.events=ChangeLedger(self.root/'reconcile_events.jsonl',load_existing)
        self.inject_remaining_failures=0
        if load_existing:
            self.read_status()
        else:
            require(not self.status_path.exists(),'调谐状态不能被覆盖')
            initial={'version':VERSION,'observed_generation':0,'desired_sha':'',
                     'cluster_epoch':store.cluster.control['epoch'],'ready':False,'live':True,
                     'reason':'Bootstrapping','mode':'unknown','finding_sha':'',
                     'first_seen_tick':None,'retry_count':0,'next_retry_tick':0,
                     'last_tick':-1,'last_success_tick':None,'last_error':'',
                     'quarantined':False,'paths':[]}
            self._save_status(initial)

    def read_status(self) -> dict:
        try:blob=json.loads(self.status_path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('状态文件缺失/损坏，必须fail-closed') from exc
        require(type(blob) is dict and set(blob)=={'status','seal'},'status封装非法')
        st=blob['status']
        require(type(st) is dict and set(st)==self.STATUS_FIELDS and blob['seal']==digest(st),
                '状态字段/摘要不匹配')
        require(st['version']==VERSION and type(st['observed_generation']) is int
                and st['observed_generation']>=0 and type(st['cluster_epoch']) is int
                and st['cluster_epoch']>=1,'状态版本或代际非法')
        require(all(type(st[k]) is bool for k in ('ready','live','quarantined')),
                '状态布尔字段非法')
        require(type(st['retry_count']) is int and st['retry_count']>=0
                and type(st['next_retry_tick']) is int and st['next_retry_tick']>=0
                and type(st['last_tick']) is int and st['last_tick']>=-1,'状态时钟/计数非法')
        require(st['first_seen_tick'] is None or
                (type(st['first_seen_tick']) is int and st['first_seen_tick']>=0),
                '漂移首次时间非法')
        require(st['last_success_tick'] is None or
                (type(st['last_success_tick']) is int and st['last_success_tick']>=0),
                '健康时间非法')
        require(type(st['paths']) is list and all(type(x) is str for x in st['paths']),
                '漂移字段列表非法')
        return st

    def _save_status(self,value:dict) -> None:
        require(set(value)==self.STATUS_FIELDS,'内部状态字段不完整')
        atomic_json(self.status_path,{'status':value,'seal':digest(value)})
        self.read_status()

    def _finish(self,previous,st,tick,action='observe'):
        st['last_tick']=tick
        self._save_status(st)
        changed=(st['reason']!=previous['reason'] or st['ready']!=previous['ready']
                 or st['observed_generation']!=previous['observed_generation'])
        if changed or action!='observe':
            self.events.add(tick,'reconcile_decision',{'generation':st['observed_generation'],
                'reason':st['reason'],'mode':st['mode'],'ready':st['ready'],
                'action':action,'finding_sha':st['finding_sha'][:16]})
        return copy.deepcopy(st)

    def reconcile_once(self,tick:int,lease:dict,crash_after_repair=False):
        self.lease.verify(lease,tick)
        prev=self.read_status()
        require(type(tick) is int and tick>=prev['last_tick'],'不能倒退逻辑tick')
        if tick==prev['last_tick']:
            return copy.deepcopy(prev)   # 同tick重复事件完全幂等；但仍核对lease。
        st=copy.deepcopy(prev)
        st['ready']=False;st['live']=True
        try:
            # 使用真实指针、两副本配置、模型权重身份与WAL控制指针。
            pointer,desired=self.store.read_active(require_healthy=False)
            require(self.store.cluster._read_control()==self.store.cluster.control,
                    '内存控制状态与持久epoch/ACK不一致')
            self.store.cluster._check_artifacts()
            report=self.store.detect_drift()
            classification=classify_drift(report['findings'],self.policy)
        except (LessonError,OSError) as exc:
            st.update(reason='ManualReview',mode='manual',paths=['untrusted_control_or_artifact'],
                      last_error=str(exc),quarantined=True,finding_sha='',first_seen_tick=tick,
                      retry_count=0,next_retry_tick=0)
            return self._finish(prev,st,tick,'block')
        basis=digest({'generation':pointer['generation'],'sha':pointer['sha'],
                      'epoch':self.store.cluster.control['epoch'],'findings':report['findings']})
        if (st['finding_sha']!=basis or st['observed_generation']!=pointer['generation']
                or st['cluster_epoch']!=self.store.cluster.control['epoch']):
            st.update(first_seen_tick=tick,retry_count=0,next_retry_tick=0,quarantined=False)
        st.update(observed_generation=pointer['generation'],desired_sha=pointer['sha'],
                  cluster_epoch=self.store.cluster.control['epoch'],finding_sha=basis,
                  paths=classification['paths'],mode=classification['mode'],last_error='')
        if classification['mode']=='healthy':
            try:self.store.assert_aligned(pointer,desired)
            except LessonError as exc:
                st.update(reason='ManualReview',mode='manual',ready=False,quarantined=True,
                          last_error=str(exc),paths=['untrusted_runtime_state'])
                return self._finish(prev,st,tick,'block')
            st.update(reason='Ready',ready=True,quarantined=False,first_seen_tick=None,
                      retry_count=0,next_retry_tick=0,last_success_tick=tick,paths=[])
            return self._finish(prev,st,tick)
        if classification['mode']=='manual':
            st.update(reason='ManualReview',ready=False,quarantined=True)
            return self._finish(prev,st,tick,'block')
        # 仅一处低风险遥测比例漂移才可进入自动修复，其余全部fail-closed。
        if st['quarantined']:
            st['reason']='Quarantined'
            return self._finish(prev,st,tick)
        if tick-st['first_seen_tick']<self.policy.grace_ticks:
            st['reason']='GracePeriod'
            return self._finish(prev,st,tick)
        if tick<st['next_retry_tick']:
            st['reason']='RetryBackoff'
            return self._finish(prev,st,tick)
        try:
            # 不授权改变模型、WAL、租户或容量；修复前核对两副本容量与权重。
            self.store.safe_capacity(desired)
            self.store.cluster.audit()
            if self.inject_remaining_failures>0:
                self.inject_remaining_failures-=1
                raise SimulatedTransientRepair('本地模拟临时I/O错误，未触碰配置')
            self.store.reconcile(classification['replica'],
                                 self.store.approval(desired,pointer['generation']),
                                 tick,pointer['generation'])
            if crash_after_repair:
                raise SimulatedCrash('模拟修复已落盘，但状态尚未提交：恢复后应重新观察')
            require(self.store.detect_drift()['healthy'],'修复后仍有漂移')
        except SimulatedTransientRepair as exc:
            st['retry_count']+=1
            if st['retry_count']>=self.policy.max_retries:
                st.update(reason='Quarantined',quarantined=True,next_retry_tick=0,
                          last_error=str(exc))
                return self._finish(prev,st,tick,'quarantine')
            wait=min(self.policy.retry_cap_ticks,self.policy.retry_base_ticks*2**(st['retry_count']-1))
            st.update(reason='RetryBackoff',next_retry_tick=tick+wait,last_error=str(exc))
            return self._finish(prev,st,tick,'backoff')
        except LessonError as exc:
            st.update(reason='ManualReview',mode='manual',ready=False,quarantined=True,
                      last_error=str(exc))
            return self._finish(prev,st,tick,'block')
        st.update(reason='AutoRepaired',ready=True,quarantined=False,first_seen_tick=None,
                  retry_count=0,next_retry_tick=0,last_success_tick=tick,paths=[])
        return self._finish(prev,st,tick,'auto_repair')

    def route(self,tick:int,tenant:str,key:str,text='hello',max_new_tokens=4) -> dict:
        st=self.read_status();p,s=self.store.read_active(require_healthy=False)
        require(type(tick) is int and tick>=0,'路由逻辑tick非法')
        require(st['ready'] and st['live'] and not st['quarantined'],
                '控制循环未宣告Ready，拒绝新请求')
        require(st['observed_generation']==p['generation'] and st['desired_sha']==p['sha']
                and st['cluster_epoch']==self.store.cluster._read_control()['epoch'],
                '状态观测代际/epoch落后于实际配置')
        require(st['last_tick']<=tick<=st['last_tick']+self.policy.status_fresh_ticks,
                '状态过期，需重新调谐后才能放行')
        self.store.assert_aligned(p,s)
        return self.store.route(tenant,key,text,max_new_tokens)

    def manual_repair(self,lease:dict,tick:int,approval:dict) -> dict:
        self.lease.verify(lease,tick)
        st=self.read_status();require(tick>st['last_tick'],'修复tick不能倒退/重放')
        p,s=self.store.read_active(False)
        findings=self.store.detect_drift()['findings']
        require(len(findings)==1 and findings[0]['status']=='drift'
                and findings[0]['replica'] in ('A','B'),
                '人工修复只适用于一处完整可解析的副本配置漂移；其他故障另行处置')
        item=findings[0]
        required={'ticket','actor','generation','config_sha','replica','finding_sha','decision'}
        require(type(approval) is dict and set(approval)==required and
                approval['decision']=='approved' and
                approval['ticket']=='INC-0084' and
                type(approval['actor']) is str and bool(approval['actor']) and
                approval['generation']==p['generation'] and approval['config_sha']==p['sha']
                and approval['replica']==item['replica'] and
                approval['finding_sha']==digest(findings),
                '显式模拟人工审批没有绑定本次现场状态')
        self.store.cluster._check_artifacts()
        self.store.safe_capacity(s)
        repaired=self.store.reconcile(item['replica'],
                                      self.store.approval(s,p['generation']),tick,p['generation'])
        self.events.add(tick,'manual_repair',{'generation':p['generation'],
                                             'replica':item['replica'],'ticket':approval['ticket'],
                                             'field':item['path']})
        result=self.reconcile_once(tick,lease)
        require(result['ready'],'人工修复后尚未达到Ready')
        return {'repair':repaired,'status':result,'simulated_approval_only':True}

    def manual_approval(self) -> dict:
        p,_=self.store.read_active(False)
        findings=self.store.detect_drift()['findings']
        require(len(findings)==1 and findings[0].get('replica') in ('A','B'),
                '仅支持对单副本配置漂移创建人工授权夹具')
        return {'ticket':'INC-0084','actor':'local_operator','generation':p['generation'],
                'config_sha':p['sha'],'replica':findings[0]['replica'],
                'finding_sha':digest(findings),'decision':'approved'}

class SimulatedTransientRepair(RuntimeError):
    pass


def finish_request(store:ConfigStore,rid:str) -> list[dict]:
    events=[]
    for _ in range(6):
        active=store.cluster.rep[store.cluster.control['active']]
        st=active.ledger.sessions[rid]
        if st.terminal is not None:break
        events.append(store.step(rid))
    require(store.cluster.rep[store.cluster.control['active']].ledger.sessions[rid].terminal is not None,
            '请求没有在预算内到达终态')
    return events

## 11. 续约与 term 围栏：举个边界例子

```text
A 在 tick 0 取得 term=1，expires=12。
A 在 tick 4 续约；term仍为1，但token更新，旧token立刻失效。
A 停止续约，在新的有效期结束前，B不得夺取term。
过期后，B取得term=2；A即使恢复执行，仍不能claim、ack、route。
```

期限端点是包含式 `[issued_tick, expires_tick]`；只有严格更晚的 tick 才能被视为过期。实现中含清晰的 `verify()` 与示例，不会把逻辑 tick 当作现实墙钟。

In [16]:
def renew_lease(lease_store:LocalLease,ticket:dict,tick:int,ttl:int=12) -> dict:
    """单进程串行续约。新的sealed票据立即取代旧票据；不是远端CAS/共识。"""
    require(type(tick) is int and tick>=0 and type(ttl) is int and 2<=ttl<=100,
            '租约续约的tick/ttl非法')
    lease_store.verify(ticket,tick)
    require(tick>=ticket['issued_tick'], '逻辑时钟倒退')
    core={'holder':ticket['holder'],'term':ticket['term'],
          'cluster_epoch':ticket['cluster_epoch'],'issued_tick':tick,
          'expires_tick':tick+ttl,'version':VERSION,
          'token':digest({'holder':ticket['holder'],'term':ticket['term'],
                           'epoch':ticket['cluster_epoch'],'issued':tick,
                           'renewed':True,'ttl':ttl})}
    result={**core,'seal':digest(core)}
    atomic_json(lease_store.path,result)
    lease_store.verify(result,tick)
    return result


def lease_identity(ticket:dict) -> dict:
    return {k:ticket[k] for k in ('holder','term','cluster_epoch','token','expires_tick')}

## 12. WorkQueue：三种任务状态与一个 dirty 标记

每条队列记录只保留一份资源 key，例如 `A`、`B`、`cluster` 和 `resync`。此例最多同时存3个唯一 key，重复watch合并，不随事件风暴无限扩张。

| 状态 | 含义 | 是否仍占队列容量 |
|---|---|---|
| `pending` | 可以领取或等待 `not_before` | 是 |
| `inflight` | 已发给某个 term，尚未 ack | 是 |
| `parked` | 人工隔离或超出重试预算 | 是 |

最容易写错：**Get/claim 不删除条目；ack 才可能移除。** 处理中又收到相同 key 的 watch，标记 `dirty=True`，此次 ack 要把 key 重新放入 pending，不能吞掉第二次通知。

client-go 的 WorkQueue 把去重、公平和不并发处理同一 key 作为关键特性；本课模拟串行版而非真实条件变量。([client-go WorkQueue API](https://pkg.go.dev/k8s.io/client-go/util/workqueue))

In [17]:
class DurableWorkQueue:
    """单进程教学持久队列：同key合并，claim不删除；完成后才ack。

    显式实现dirty/processing串行状态，但不提供client-go并发条件变量或跨进程互斥。
    需要一个串行化的权威存储才能把此设计带到生产环境。
    """
    KEYS=frozenset({'A','B','cluster','resync'})
    MAX_HISTORY=256

    def __init__(self,root:Path,capacity:int=3,load_existing:bool=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.path=self.root/'queue.json'
        self.events=ChangeLedger(self.root/'queue_events.jsonl',load_existing)
        if load_existing:
            self.read()
        else:
            require(type(capacity) is int and 1<=capacity<=4,'队列容量需要1..4')
            require(not self.path.exists(),'拒绝覆盖既存队列')
            self._save({'version':VERSION,'capacity':capacity,'clock':-1,'next_seq':1,
                        'entries':[],'overflow':False,'overflow_count':0,
                        'coalesced_events':0,'accepted_events':0,'processed':0,
                        'retries':0,'reclaims':0,'parked_count':0,'redrives':0,
                        'full_resync_ready':False})

    def _save(self,data:dict):
        require(data['version']==VERSION,'队列版本不匹配')
        atomic_json(self.path,{'data':data,'seal':digest(data)})

    def read(self):
        try:obj=json.loads(self.path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('队列不可读取，停止所有控制写入') from exc
        require(type(obj) is dict and set(obj)=={'data','seal'},'队列封装非法')
        d=obj['data']
        require(type(d) is dict and set(d)=={'version','capacity','clock','next_seq',
               'entries','overflow','overflow_count','coalesced_events','accepted_events',
               'processed','retries','reclaims','parked_count','redrives','full_resync_ready'},'队列字段不匹配')
        require(obj['seal']==digest(d) and d['version']==VERSION,'队列摘要或版本错误')
        require(type(d['capacity']) is int and 1<=d['capacity']<=4,'队列容量非法')
        require(type(d['clock']) is int and d['clock']>=-1 and type(d['next_seq']) is int
                and d['next_seq']>=1,'队列逻辑时间/序号非法')
        require(type(d['overflow']) is bool and type(d['full_resync_ready']) is bool,
                'overflow/full_resync状态非法')
        for key in ('overflow_count','coalesced_events','accepted_events','processed',
                    'retries','reclaims','parked_count','redrives'):
            require(type(d[key]) is int and d[key]>=0,'队列计数非法：'+key)
        require(type(d['entries']) is list and len(d['entries'])<=d['capacity'],
                '队列条目超过容量')
        seqs=set();keys=set()
        for item in d['entries']:
            require(type(item) is dict and set(item)=={'key','generation','sha','epoch','seq',
                    'reasons','not_before','attempts','phase','claim','dirty'},'队列条目字段错误')
            require(item['key'] in self.KEYS and item['key'] not in keys,'非法/重复资源key')
            keys.add(item['key'])
            require(type(item['generation']) is int and item['generation']>=1
                    and type(item['epoch']) is int and item['epoch']>=1
                    and type(item['seq']) is int and 1<=item['seq']<d['next_seq']
                    and item['seq'] not in seqs,'资源版本/seq非法')
            seqs.add(item['seq'])
            require(type(item['sha']) is str and bool(re.fullmatch('[0-9a-f]{64}',item['sha'])),
                    '资源配置哈希非法')
            require(type(item['reasons']) is list and bool(item['reasons']) and
                    all(type(x) is str and bool(re.fullmatch('[a-z_]{2,24}',x)) for x in item['reasons'])
                    and item['reasons']==sorted(set(item['reasons'])),'理由字段非规范化')
            require(all(type(item[x]) is int and item[x]>=0 for x in ('not_before','attempts')),
                    '重试计数/时间非法')
            require(item['phase'] in {'pending','inflight','parked'},'队列状态非法')
            require(type(item['dirty']) is bool and (item['phase']=='inflight' or not item['dirty']),
                    'dirty只允许在处理中的资源被再次通知时出现')
            if item['phase']=='inflight':
                claim=item['claim']
                require(type(claim) is dict and set(claim)=={'holder','term','epoch','nonce'},
                        'inflight缺少凭据')
                require(type(claim['holder']) is str and
                        bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',claim['holder'])) and
                        type(claim['term']) is int and claim['term']>=1 and
                        type(claim['epoch']) is int and claim['epoch']>=1 and
                        type(claim['nonce']) is str and bool(re.fullmatch('[0-9a-f]{64}',claim['nonce'])),
                        'claim字段错误')
            else:require(item['claim'] is None,'非inflight不应保留claim')
        return copy.deepcopy(d)

    @staticmethod
    def _item(d,key):
        return next((x for x in d['entries'] if x['key']==key),None)

    def _clock(self,d,tick):
        require(type(tick) is int and 0<=tick<=100000,'逻辑tick非法')
        require(tick>=d['clock'],'队列操作不能倒退时钟')
        d['clock']=tick

    def enqueue(self,key:str,revision:dict,tick:int,reason:str='watch') -> dict:
        """全队列时不丢弃资源变更：设置持久overflow并明确拒绝；必须完整resync才能解锁。"""
        d=self.read();self._clock(d,tick)
        require(key in self.KEYS,'未知资源key')
        require(bool(re.fullmatch('[a-z_]{2,24}',reason)),'reason非法')
        require(type(revision) is dict and set(revision)=={'generation','sha','epoch'},
                '期望版本必须同时包含代际、内容哈希、cluster epoch')
        require(type(revision['generation']) is int and revision['generation']>=1 and
                type(revision['epoch']) is int and revision['epoch']>=1 and
                type(revision['sha']) is str and bool(re.fullmatch('[0-9a-f]{64}',revision['sha'])),
                '期望版本身份非法')
        old=self._item(d,key)
        if old:
            require(revision['generation']>=old['generation'],'旧代际事件不能倒退覆盖新事件')
            if revision['generation']==old['generation']:
                require(revision['sha']==old['sha'] and revision['epoch']==old['epoch'],
                        '同代际出现不同内容/epoch，不能吞掉冲突')
                d['coalesced_events']+=1
                old['reasons']=sorted(set(old['reasons']+[reason]))
                if old['phase']=='inflight':old['dirty']=True
                self._save(d)
                return {'coalesced':True,'key':key,'seq':old['seq'],'size':len(d['entries'])}
            # 更高代际的事件覆盖旧意图，但仍按最新磁盘事实调谐；原inflight凭据作废。
            if old['phase']=='inflight':
                d['overflow']=True;d['full_resync_ready']=False;d['overflow_count']+=1;self._save(d)
                self.events.add(tick,'queue_overflow',{'key':key,'reason':'inflight_new_generation'})
                raise LessonError('更高代际发生于同key的inflight处理期间，拒绝并标记完整resync')
            old.update(generation=revision['generation'],sha=revision['sha'],
                       epoch=revision['epoch'],reasons=sorted(set(old['reasons']+[reason])),
                       not_before=tick,attempts=0,phase='pending',claim=None,dirty=False)
            self._save(d)
            return {'coalesced':True,'new_generation':True,'key':key,'seq':old['seq']}
        if len(d['entries'])>=d['capacity']:
            d['overflow']=True;d['full_resync_ready']=False;d['overflow_count']+=1;self._save(d)
            self.events.add(tick,'queue_overflow',{'key':key,'reason':'capacity'})
            raise LessonError('工作队列已满：持久标记needs_full_resync，阻断路由，不能静默丢事件')
        item={'key':key,'generation':revision['generation'],'sha':revision['sha'],
              'epoch':revision['epoch'],'seq':d['next_seq'],'reasons':[reason],
              'not_before':tick,'attempts':0,'phase':'pending','claim':None,'dirty':False}
        d['next_seq']+=1;d['entries'].append(item);d['accepted_events']+=1
        self._save(d)
        self.events.add(tick,'queue_enqueue',{'key':key,'seq':item['seq'],
                                             'generation':revision['generation']})
        return {'coalesced':False,'key':key,'seq':item['seq'],'size':len(d['entries'])}

    def claim(self,ticket:dict,tick:int,lease_store:LocalLease) -> dict|None:
        lease_store.verify(ticket,tick)
        d=self.read();self._clock(d,tick)
        candidates=[x for x in d['entries'] if x['phase']=='pending' and x['not_before']<=tick]
        if not candidates:
            self._save(d)
            return None
        # FIFO per key；延迟重试不阻塞其他ready资源；不在此读路由内容。
        item=min(candidates,key=lambda x:x['seq'])
        claim={'holder':ticket['holder'],'term':ticket['term'],'epoch':ticket['cluster_epoch'],
               'nonce':digest({'seq':item['seq'],'key':item['key'],'term':ticket['term'],
                               'ticket':ticket['token'],'tick':tick})}
        item['phase']='inflight';item['claim']=claim
        self._save(d)
        self.events.add(tick,'queue_claim',{'key':item['key'],'seq':item['seq'],
                                           'term':ticket['term']})
        return {'key':item['key'],'revision':{k:item[k] for k in ('generation','sha','epoch')},
                'seq':item['seq'],'claim':claim,'attempts':item['attempts']}

    def _claimed(self,claim:dict,ticket:dict,tick:int,lease_store:LocalLease):
        lease_store.verify(ticket,tick)
        d=self.read();self._clock(d,tick)
        require(type(claim) is dict and set(claim)=={'holder','term','epoch','nonce'},
                'ack/retry必须持有完整claim')
        require(claim['holder']==ticket['holder'] and claim['term']==ticket['term']
                and claim['epoch']==ticket['cluster_epoch'], '控制者term与claim不一致')
        match=[x for x in d['entries'] if x['phase']=='inflight' and x['claim']==claim]
        require(len(match)==1,'任务不是当前控制者的唯一inflight；旧ack被拒绝')
        return d,match[0]

    def ack(self,claim:dict,ticket:dict,tick:int,lease_store:LocalLease) -> dict:
        d,item=self._claimed(claim,ticket,tick,lease_store)
        key=item['key'];d['processed']+=1
        if item['dirty']:
            # 处理过程中又收到Add，同key必须再次被Get，不能丢失边沿。
            item.update(phase='pending',claim=None,dirty=False,attempts=0,
                        not_before=tick+1,seq=d['next_seq'])
            d['next_seq']+=1;d['redrives']+=1;requeued=True
        else:
            d['entries'].remove(item);requeued=False
            if key=='resync':d['full_resync_ready']=True
        self._save(d)
        self.events.add(tick,'queue_ack',{'key':key,'requeued_due_to_dirty':requeued})
        return {'key':key,'remaining':len(d['entries']),'dirty_requeued':requeued}

    def retry(self,claim:dict,ticket:dict,tick:int,lease_store:LocalLease,
              delay:int=1,cap:int=8,max_attempts:int=4,exponential:bool=True) -> dict:
        require(type(delay) is int and delay>=1 and type(cap) is int and cap>=1
                and type(max_attempts) is int and max_attempts>=1,'重试预算非法')
        d,item=self._claimed(claim,ticket,tick,lease_store)
        item['attempts']+=1;d['retries']+=1
        if item['attempts']>=max_attempts:
            item.update(phase='parked',claim=None,not_before=tick,dirty=False)
            d['parked_count']+=1;status='parked'
        else:
            wait=min(cap,max(delay,2**(item['attempts']-1 if exponential else 0)))
            item.update(phase='pending',claim=None,not_before=tick+wait,dirty=False)
            status='retry'
        self._save(d)
        self.events.add(tick,'queue_retry',{'key':item['key'],'status':status,
                                             'attempts':item['attempts']})
        return {'key':item['key'],'status':status,'attempts':item['attempts'],
                'not_before':item['not_before']}

    def park(self,claim:dict,ticket:dict,tick:int,lease_store:LocalLease) -> dict:
        d,item=self._claimed(claim,ticket,tick,lease_store)
        item.update(phase='parked',claim=None,dirty=False)
        d['parked_count']+=1
        self._save(d)
        self.events.add(tick,'queue_park',{'key':item['key']})
        return {'key':item['key'],'parked':True}

    def recover_orphans(self,ticket:dict,tick:int,lease_store:LocalLease) -> dict:
        lease_store.verify(ticket,tick)
        d=self.read();self._clock(d,tick)
        count=0
        for item in d['entries']:
            if item['phase']=='inflight':
                old=item['claim']
                require(old['term']<ticket['term'] and old['epoch']==ticket['cluster_epoch'],
                        '不能抢占仍属于当前term的任务，或偷偷跨cluster epoch')
                item.update(phase='pending',claim=None,not_before=tick,dirty=False)
                count+=1
        d['reclaims']+=count;self._save(d)
        if count:self.events.add(tick,'queue_reclaim',{'count':count,'term':ticket['term']})
        return {'reclaimed':count,'term':ticket['term']}

    def release_parked(self,key:str,ticket:dict,tick:int,lease_store:LocalLease) -> dict:
        lease_store.verify(ticket,tick)
        d=self.read();self._clock(d,tick)
        item=self._item(d,key)
        require(item is not None and item['phase']=='parked','只有显式parked资源可重排')
        item.update(phase='pending',claim=None,attempts=0,not_before=tick,dirty=False)
        self._save(d)
        self.events.add(tick,'queue_unpark',{'key':key})
        return {'key':key,'unparked':True}

    def clear_overflow_after_resync(self,ticket:dict,tick:int,lease_store:LocalLease,
                                    store:ConfigStore,loop:ReconcileLoop) -> bool:
        lease_store.verify(ticket,tick)
        d=self.read();self._clock(d,tick)
        require(all(item['phase']!='inflight' for item in d['entries']),
                '还有未提交的inflight，不可宣告完整resync')
        require(not d['entries'],'队列还有待处理/隔离任务，不可解除溢出状态')
        require(d['overflow'] and d['full_resync_ready'],
                '必须先成功处理本次溢出后的resync任务，不能凭队列清空自行解锁')
        require(loop.read_status()['ready'] and store.detect_drift()['healthy'],
                '完整resync必须真实健康且Ready')
        store.assert_aligned()
        d['overflow']=False;d['full_resync_ready']=False;self._save(d)
        self.events.add(tick,'queue_resync_clear',{'generation':store.read_pointer()['generation']})
        return True

    def stats(self):
        d=self.read()
        return {'capacity':d['capacity'],'depth':len(d['entries']),
                'pending':sum(x['phase']=='pending' for x in d['entries']),
                'inflight':sum(x['phase']=='inflight' for x in d['entries']),
                'parked':sum(x['phase']=='parked' for x in d['entries']),
                'overflow':d['overflow'],'overflow_count':d['overflow_count'],
                'full_resync_ready':d['full_resync_ready'],
                'accepted_events':d['accepted_events'],'coalesced_events':d['coalesced_events'],
                'processed':d['processed'],'retries':d['retries'],'reclaims':d['reclaims'],
                'redrives':d['redrives']}

## 13. 练一个不用语言模型的队列小实验

先用两个不同资源手算：A 失败推迟处理，B 不应该因此被阻塞。同一个 A 在 inflight 期间再收到 watch，完成后应再次入队。

下面的 `DurableWorkQueue` 只处理资源键，建在临时目录，**绝不向服务发送真实请求**；完整 end-to-end 实验在后面另行运行。

In [18]:
with tempfile.TemporaryDirectory() as tmp:
    demo_q = DurableWorkQueue(Path(tmp)/'queue',capacity=2)
    revision={'generation':1,'sha':'0'*64,'epoch':1}
    print('首个通知:',demo_q.enqueue('A',revision,0,'watch'))
    print('重复通知:',demo_q.enqueue('A',revision,0,'periodic'))
    print('第二个key:',demo_q.enqueue('B',revision,0,'watch'))
    print('现在的容量和去重:',demo_q.stats())
    try:
        demo_q.enqueue('cluster',revision,0,'watch')
    except LessonError as e:
        print('满队列明确拒绝:',str(e))
    print('overflow持久标志:',demo_q.stats()['overflow'])

首个通知: {'coalesced': False, 'key': 'A', 'seq': 1, 'size': 1}
重复通知: {'coalesced': True, 'key': 'A', 'seq': 1, 'size': 1}
第二个key: {'coalesced': False, 'key': 'B', 'seq': 2, 'size': 2}
现在的容量和去重: {'capacity': 2, 'depth': 2, 'pending': 2, 'inflight': 0, 'parked': 0, 'overflow': False, 'overflow_count': 0, 'full_resync_ready': False, 'accepted_events': 2, 'coalesced_events': 1, 'processed': 0, 'retries': 0, 'reclaims': 0, 'redrives': 0}
满队列明确拒绝: 工作队列已满：持久标记needs_full_resync，阻断路由，不能静默丢事件
overflow持久标志: True


## 14. 失败重试与宽限不是一种事件

如果下游暂时故障，允许有限次延迟重试，例如 `1 → 2 → 4 → 8` tick（最多8）。但若只是配置漂移的**稳定观察宽限**，每次重新观察等1个tick；不应因为观察宽限次数增加就误放大指数退避。

`parked` 不等于“丢弃任务”；它继续占有容量与路由安全门槛，必须有显式处理。不能让无限重试把模型服务变成修复风暴。[client-go 对 `AddRateLimited` 与 `Forget` 的说明](https://pkg.go.dev/k8s.io/client-go/util/workqueue)

当资源 key 的上一轮处理失败，调度器仍可领取另一个已就绪 key。这比一个全局睡眠并阻塞所有资源更合理。

In [19]:
print('失败退避:',[answer_b(n) for n in range(1,7)] if 'answer_b' in globals() else [min(8,2**(n-1)) for n in range(1,7)])
print('宽限重新观察:',[1]*5)
print('警告：时间单位是逻辑tick，没有线程休眠或真实Kubernetes队列。')

失败退避: [1, 2, 4, 8, 8, 8]
宽限重新观察: [1, 1, 1, 1, 1]
警告：时间单位是逻辑tick，没有线程休眠或真实Kubernetes队列。


## 15. 溢出：拒绝事件不能被悄悄忘掉

当3个槽已满，第四个独立资源事件到达，不能静默丢弃后却继续路由。代码会把 `overflow=True` **持久落盘**并拒绝这一通知。即使稍后把先前3个任务处理完，也不能立刻清除标记，因为第四个事件的具体状态可能从未被观察。

完整解锁必须同时满足：

- 不再有未提交、延迟或隔离任务；
- 新的 `resync` 任务确实被领取并完成；
- 真正重新读取配置，状态 Ready、双副本一致、模型/WAL/KV检查通过。

这里的 `full_resync_ready` 只表示本次溢出后的**成功 resync 证据**，解锁后立即消耗，不允许无限次复用。

## 16. 两个控制者：谁有权写？谁只能观察？

`controller_a` 和 `controller_b` 都能读队列及当前租约，也都能提交有界 watch 提示；但是只有持有**当前有效票据**的控制者可以 claim、ack、发起调谐或放行新请求。

`take_over()` 不只是把变量 `leader='B'` 改掉。必须等 Lease 过期并让 term 递增，重新核验 cluster epoch，再把旧 term 的 inflight 恢复为待处理；新 Leader 随后**重读磁盘状态**，确认刚才的外部修复究竟成功没有。

如果旧控制者在修复落盘后、队列 ack 前崩溃，恢复策略是**至少一次重调谐＋幂等外部写入**；不会宣称端到端 exactly-once。

In [20]:
class ControllerWorker:
    def __init__(self,name:str,loop:ReconcileLoop,queue:DurableWorkQueue):
        require(bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',name)),'控制者名称非法')
        self.name=name;self.loop=loop;self.queue=queue
        self.ticket=None
        self.last_result=None

    def elect(self,tick:int,ttl:int=12):
        self.ticket=self.loop.lease.acquire(self.name,tick,ttl)
        return lease_identity(self.ticket)

    def heartbeat(self,tick:int,ttl:int=12):
        require(self.ticket is not None and self.ticket['holder']==self.name,'无持有者票据')
        self.ticket=renew_lease(self.loop.lease,self.ticket,tick,ttl)
        return lease_identity(self.ticket)

    def _verify(self,tick:int):
        require(self.ticket is not None and self.ticket['holder']==self.name,'Follower不能写控制状态')
        self.loop.lease.verify(self.ticket,tick)

    def revision(self):
        p,_=self.loop.store.read_active(require_healthy=False)
        return {'generation':p['generation'],'sha':p['sha'],
                'epoch':self.loop.store.cluster._read_control()['epoch']}

    def observe(self):
        # Follower可读取，不应隐式刷新Status/续约或触发修复。
        return {'status':self.loop.read_status(),'queue':self.queue.stats(),
                'lease':self.loop.lease.read()}

    def notify(self,key:str,tick:int,reason='watch'):
        # 通知端不依赖leader，但不得包含或替代权威最新配置；工作者实际读取磁盘。
        return self.queue.enqueue(key,self.revision(),tick,reason)

    def process_one(self,tick:int,crash_after_repair=False) -> dict:
        self._verify(tick)
        claim=self.queue.claim(self.ticket,tick,self.loop.lease)
        if claim is None:return {'worker':self.name,'tick':tick,'action':'idle'}
        # 不把事件里的配置当目标，也不认可已无效的代际快照。
        latest=self.revision()
        if claim['revision']!=latest:
            # 当旧事件已被处理，但版本变化，必须重新观察。失败闭合而非覆盖新目标。
            self.queue.park(claim['claim'],self.ticket,tick,self.loop.lease)
            return {'worker':self.name,'tick':tick,'action':'stale_parked','key':claim['key']}
        try:
            st=self.loop.reconcile_once(tick,self.ticket,crash_after_repair=crash_after_repair)
        except SimulatedCrash:
            # 关键点：claim仍inflight，恢复不能假定实际写入已提交/未提交。
            raise
        self._verify(tick)
        if st['ready']:
            completed=self.queue.ack(claim['claim'],self.ticket,tick,self.loop.lease)
            action='ack'
        elif st['reason'] in {'GracePeriod','RetryBackoff'}:
            next_tick=st['next_retry_tick'] if st['reason']=='RetryBackoff' else tick+1
            completed=self.queue.retry(claim['claim'],self.ticket,tick,self.loop.lease,
                  delay=max(1,next_tick-tick),exponential=(st['reason']=='RetryBackoff'))
            action=completed['status']
        else:
            completed=self.queue.park(claim['claim'],self.ticket,tick,self.loop.lease)
            action='park'
        self.last_result={'worker':self.name,'tick':tick,'action':action,
                          'key':claim['key'],'ready':st['ready'],'reason':st['reason'],
                          'details':completed}
        return copy.deepcopy(self.last_result)

    def take_over(self,tick:int,ttl:int=12) -> dict:
        ticket=self.elect(tick,ttl)
        restored=self.queue.recover_orphans(self.ticket,tick,self.loop.lease)
        return {'ticket':ticket,**restored}

    def route(self,tick:int,tenant:str,key:str,text='hello',max_new_tokens=4) -> dict:
        self._verify(tick)
        stats=self.queue.stats()
        require(not stats['overflow'] and stats['depth']==0,
                '工作队列不干净或曾溢出：停止新请求直到完整resync')
        return self.loop.route(tick,tenant,key,text,max_new_tokens)


def finish_request(store:ConfigStore,rid:str) -> list[dict]:
    events=[]
    for _ in range(6):
        active=store.cluster.rep[store.cluster.control['active']]
        st=active.ledger.sessions[rid]
        if st.terminal is not None:break
        events.append(store.step(rid))
    require(store.cluster.rep[store.cluster.control['active']].ledger.sessions[rid].terminal is not None,
            '请求没有在预算内到达终态')
    return events


def make_demo(root:Path,seed:int=8501,queue_capacity:int=3,load_existing=False):
    root=Path(root)
    if load_existing:
        tok,models=load_model_artifacts(root/'cluster'/'models.pt')
        control=json.loads((root/'cluster'/'control.json').read_text(encoding='utf-8'))
        cluster=Coordinator(root/'cluster',tok,models,policy=control['policy'],
              seed=control['seed'],capacity_blocks=control['capacity_blocks'],load_existing=True)
        store=ConfigStore(root/'config',cluster,load_existing=True)
        loop=ReconcileLoop(root/'controller',store,load_existing=True)
        queue=DurableWorkQueue(root/'workqueue',load_existing=True)
    else:
        tok,models=build_servable_models(seed)
        cluster=Coordinator(root/'cluster',tok,models,policy='sync',seed=seed,capacity_blocks=64)
        store=ConfigStore(root/'config',cluster)
        loop=ReconcileLoop(root/'controller',store)
        queue=DurableWorkQueue(root/'workqueue',capacity=queue_capacity)
    alpha=ControllerWorker('controller_a',loop,queue)
    beta=ControllerWorker('controller_b',loop,queue)
    return tok,models,cluster,store,loop,queue,alpha,beta


def edit_replica_field(store:ConfigStore,name:str,**changes):
    p,desired=store.read_active(False)
    bad=changed_config(desired,**changes)
    atomic_json(store._replica_path(name,p['generation']),bad.wire())
    return {'replica':name,'generation':p['generation'],
            'changed_fields':sorted(changes),'observed_sha':digest(bad.wire())}

## 17. 完整事故链：同一模型上的真实推理与真实恢复

默认实验包含以下顺序：

1. 48条相同资源 watch 合并成一个队列项，A正常调谐并用真实CPU KV产生流式token。
2. 一处受控遥测配置漂移经历两次宽限重排，A自动修复；另一处租户权限漂移进入人工隔离。
3. 三个槽满时第4个资源显式拒绝，阻断新请求；清空后仍要完成全量resync。
4. A在修复配置后、提交队列 ACK 前崩溃，任务保持inflight。
5. 旧Lease过期后B取得新term，回收旧claim，重读磁盘发现实际修复已成功，**不重复修复**。
6. 在新控制者下继续真实推理、WAL commit 和终态 KV 释放，冷启动再审计。

**没有后台服务、真实线程竞争或远端集群。** 相同输入和随机种子在同CPU环境下复现；不同Torch版本/设备可能有数值差异。

In [21]:
def run_experiment(output_dir:Path=Path('lesson_85_outputs'),seed:int=8501) -> Path:
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    root=Path(output_dir)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    root.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'seed':seed,'status':'running',
              'python':platform.python_version(),'torch':str(torch.__version__),
              'device':'cpu','sources':SOURCES,
              'limitations':['local_single_process_serialized_file_storage_not_consensus',
                 'two_logical_workers_not_parallel_processes',
                 'ticks_not_real_time','no_Kubernetes_API','no_real_IAM',
                 'SHA_not_authenticated_signature','CPU_KV_tensors_not_GPU_latency',
                 'events_are_hints_reconcile_latest_on_disk','at_least_once_not_exactly_once']}
    write_json(root/'manifest.json',manifest)
    report={}
    try:
        tok,models,cluster,store,loop,queue,a,b=make_demo(root,seed)
        t0=a.elect(0,12)
        storm=[a.notify('cluster',0,'watch') for _ in range(48)]
        require(sum(not x['coalesced'] for x in storm)==1 and queue.stats()['depth']==1,
                '48个重复通知应仅占一个队列槽')
        require(queue.stats()['coalesced_events']==47,'需要记录去重数量')
        initial=a.process_one(0)
        require(initial['action']=='ack' and initial['ready'],'Leader没有完成初始化调谐')
        req=a.route(0,'alpha','after_ready','hello',4)
        output=finish_request(store,req['rid'])
        require(bool(output) and cluster.rep['A'].pool.audit()['live_blocks']==0,
                '真实WAL/KV请求必须最终释放物理缓存')
        report['storm']={'events':48,'unique_queue_items':1,'coalesced':47,
                         'leader':t0,'worker_action':initial,'inference_version':req['version'],
                         'generated_events':len(output),'kv':store.probe_kv()}
        # 低风险漂移: 重复watch不会重置宽限计时；每次成功独占claim，按tick延迟重试。
        edit_replica_field(store,'B',telemetry_sample_percent=80)
        for _ in range(20):a.notify('B',1,'watch')
        timeline=[]
        for tick in (1,2,3):timeline.append(a.process_one(tick))
        require([x['action'] for x in timeline]==['retry','retry','ack'],
                '低风险漂移需经历两轮宽限再恢复')
        require([x['reason'] for x in timeline]==['GracePeriod','GracePeriod','AutoRepaired'],
                '相同key重试必须读取真实最新状态')
        require(store.detect_drift()['healthy'],'受控自动修复后仍有漂移')
        report['safe_reconcile']={'timeline':timeline,'merged_events':19,
                                  'queue':queue.stats()}
        old_ticket=copy.deepcopy(a.ticket)
        new_ticket=a.heartbeat(4,12)
        try:loop.reconcile_once(4,old_ticket)
        except LessonError:old_ticket_fenced=True
        else:raise AssertionError('续约后的旧票据必须失效，即使term不变')
        report['renew']={'old_ticket_fenced':old_ticket_fenced,
                         'same_term':old_ticket['term']==new_ticket['term'],
                         'expires_tick':new_ticket['expires_tick']}
        # 权限漂移：进入人工隔离，不进行低风险修复。真实批准后显式重新排队。
        edit_replica_field(store,'B',allowed_tenants=['alpha'])
        a.notify('B',5,'watch')
        danger=a.process_one(5)
        require(danger['action']=='park' and danger['reason']=='ManualReview' and
                queue.stats()['parked']==1,'权限漂移必须进入人工队列')
        try:a.route(5,'alpha','must_block','hello',4)
        except LessonError:blocked=True
        else:raise AssertionError('parked事件不能放行流量')
        approval=loop.manual_approval()
        manual=loop.manual_repair(a.ticket,6,approval)
        require(manual['status']['ready'] and store.detect_drift()['healthy'],
                '显式人工干预后应恢复Ready')
        queue.release_parked('B',a.ticket,6,loop.lease)
        post_manual=a.process_one(7)
        require(post_manual['action']=='ack' and queue.stats()['depth']==0,'人工处理后必须清理队列')
        report['danger']={'quarantined':danger,'route_blocked':blocked,
                          'manual_approval_kind':'simulated_local_only',
                          'requeued_after_manual':post_manual['action']}
        # 不丢弃第四个独立资源的通知：持久overflow，直到下一次全量观察才清除。
        for key in ('A','B','cluster'):a.notify(key,8,'watch')
        try:a.notify('resync',8,'watch')
        except LessonError:overflow_rejected=True
        else:raise AssertionError('满队列必须拒绝并持久记账')
        require(queue.stats()['overflow'] and queue.stats()['depth']==3,
                '队列满需要显式溢出状态')
        try:a.route(8,'alpha','overflow_block','hello',4)
        except LessonError:overflow_blocks=True
        else:raise AssertionError('发生队列溢出时新请求不能静默继续')
        cleared=[]
        for tick in (8,9,10):cleared.append(a.process_one(tick))
        require(all(x['action']=='ack' for x in cleared) and queue.stats()['depth']==0,
                '旧队列条目处理失败')
        require(queue.stats()['overflow'],'只清空队列不能宣称已完整观察被拒的变更')
        a.notify('resync',11,'full_resync')
        synced=a.process_one(11)
        require(synced['action']=='ack','全量观察任务失败')
        queue.clear_overflow_after_resync(a.ticket,11,loop.lease,store,loop)
        require(not queue.stats()['overflow'],'通过真实健康检查后才可清除溢出')
        after_overflow=a.route(11,'beta','after_resync','hello',4)
        finish_request(store,after_overflow['rid'])
        report['overflow']={'explicit_rejection':overflow_rejected,'blocked_routing':overflow_blocks,
                            'cleared_after_queue_empty_only':False,
                            'full_resync_action':synced['action'],'safe_after_full_resync':True,
                            'max_queue_depth':3,'queue':queue.stats()}
        # 核心崩溃点：修复已落盘，ack尚未提交。新Leader必须重新读最新状态。
        edit_replica_field(store,'B',telemetry_sample_percent=75)
        a.notify('B',12,'watch')
        wait1=a.process_one(12);wait2=a.process_one(13)
        require([wait1['action'],wait2['action']]==['retry','retry'],'修复应等待宽限')
        before=store.audit.verify()['count']
        try:a.process_one(14,crash_after_repair=True)
        except SimulatedCrash:crashed=True
        else:raise AssertionError('需要注入修复后、队列ack前崩溃')
        require(queue.stats()['inflight']==1 and store.detect_drift()['healthy'],
                '物理修复完成但未确认的inflight应持久保留')
        after=store.audit.verify()['count']
        require(after==before+1,'修复应真实落盘且恰好一次')
        try:b.elect(15,12)
        except LessonError:early_takeover_rejected=True
        else:raise AssertionError('旧租约仍有效，不能提前接管')
        require(early_takeover_rejected,'必须拒绝重叠leader')
        promoted=b.take_over(17,12)
        require(promoted['term']==2 and promoted['reclaimed']==1,
                '租约到期后的新Leader应收回旧term的inflight')
        try:a.process_one(17)
        except LessonError:old_leader_fenced=True
        else:raise AssertionError('旧Leader不能继续消费')
        recovered=b.process_one(17)
        require(recovered['action']=='ack' and recovered['reason']=='Ready',
                '新Leader必须依据磁盘最新事实恢复')
        require(store.audit.verify()['count']==after,
                '恢复不能重做已经成功落盘的修复')
        report['takeover']={'crashed_after_repair':crashed,
                            'early_takeover_rejected':early_takeover_rejected,
                            'old_leader_fenced':old_leader_fenced,
                            'old_term':old_ticket['term'],'new_term':promoted['term'],
                            'reclaimed_inflight':promoted['reclaimed'],
                            'repaired_once':store.audit.verify()['count']==after,
                            'recovered_result':recovered}
        req=b.route(17,'alpha','after_takeover','hello',4)
        finish_request(store,req['rid'])
        prev=copy.deepcopy(b.ticket)
        renewed=b.heartbeat(19,12)
        try:loop.reconcile_once(19,prev)
        except LessonError:stale_after_renew=True
        else:raise AssertionError('Leader续约前的票据仍然有效')
        require(stale_after_renew and renewed['term']==promoted['term'],
                'term在续约期间保持不变，但旧票据失效')
        req2=b.route(19,'beta','after_renew','hello',4)
        finish_request(store,req2['rid'])
        final_kv=store.probe_kv()
        require(final_kv['max_cache_full_diff']<2e-5,'增量KV与全前向不一致')
        live={name:cluster.rep[name].pool.audit()['live_blocks'] for name in ('A','B')}
        require(live=={'A':0,'B':0},'所有真实KV页必须释放')
        report['final']={'generation':store.read_pointer()['generation'],
                         'epoch':cluster.control['epoch'],'lease_term':b.ticket['term'],
                         'lease_holder':b.ticket['holder'],
                         'renew_fenced_stale_ticket':stale_after_renew,
                         'config_audit_events':store.audit.verify()['count'],
                         'controller_audit_events':loop.events.verify()['count'],
                         'wal_ack_seq':cluster.control['ack_seq'],
                         'queue':queue.stats(),'real_kv':final_kv,
                         'live_kv_blocks':live,'status':loop.read_status(),
                         'no_remote_consensus':True,'ticks_not_wall_clock':True}
        write_json(root/'experiment.json',report)
        reloaded=reload_run(root)
        write_json(root/'reload.json',reloaded)
        manifest['status']='completed';write_json(root/'manifest.json',manifest)
        print('去重48→1；安全修复与人工隔离；溢出完整resync；旧主围栏；崩溃后仅恢复一次',flush=True)
        print('真实KV在终态后归零：',live,'；队列：',queue.stats(),flush=True)
        print('输出目录：',root,flush=True)
        return root
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(root/'manifest.json',manifest)
        raise


def reload_run(folder:Path) -> dict:
    root=Path(folder)
    tok,models,cluster,store,loop,queue,a,b=make_demo(root,load_existing=True)
    control=cluster.audit()
    p,s=store.read_active(require_healthy=True)
    st=loop.read_status();lease=loop.lease.read()
    require(st['ready'] and st['observed_generation']==p['generation']
            and st['desired_sha']==p['sha'] and st['cluster_epoch']==control['epoch'],
            '冷恢复的Status与活动配置/epoch不一致')
    require(lease['cluster_epoch']==control['epoch'],'冷恢复lease与集群epoch不一致')
    require(queue.stats()['depth']==0 and not queue.stats()['overflow'],
            '队列仍有未确认任务或溢出事件，不能宣告恢复Ready')
    require(store.detect_drift()['healthy'],'配置漂移没有恢复')
    kv=store.probe_kv()
    require(kv['max_cache_full_diff']<2e-5,'真实KV恢复后无法匹配全前向')
    physical={name:cluster.rep[name].pool.audit()['live_blocks'] for name in ('A','B')}
    require(physical=={'A':0,'B':0},'实际物理KV仍有残留')
    return {'generation':p['generation'],'config_sha':p['sha'],
            'epoch':control['epoch'],'lease_holder':lease['holder'],
            'lease_term':lease['term'],'status_ready':st['ready'],
            'reconciliation_audit_head':loop.events.verify()['head'],
            'config_audit_head':store.audit.verify()['head'],
            'workqueue_audit_head':queue.events.verify()['head'],
            'queue':queue.stats(),'physical_KV_blocks':physical,
            'real_KV_full_logits_diff':kv['max_cache_full_diff'],
            'wal_ack_seq':control['ack_seq'],
            'not_real_distributed_consensus':True}

In [22]:
RUN_FULL_EXPERIMENT = True
EXPERIMENT_DIR = None
if RUN_FULL_EXPERIMENT:
    EXPERIMENT_DIR = run_experiment(Path('lesson_85_outputs'),seed=8501)
else:
    print('完整真实KV/WAL演练明确跳过；不会伪造报告。')

去重48→1；安全修复与人工隔离；溢出完整resync；旧主围栏；崩溃后仅恢复一次


真实KV在终态后归零： {'A': 0, 'B': 0} ；队列： {'capacity': 3, 'depth': 0, 'pending': 0, 'inflight': 0, 'parked': 0, 'overflow': False, 'overflow_count': 1, 'full_resync_ready': False, 'accepted_events': 8, 'coalesced_events': 66, 'processed': 8, 'retries': 4, 'reclaims': 1, 'redrives': 0}


输出目录： lesson_85_outputs/run_20261105_094338_567278Z


## 18. 实际结果：区分“通知次数”“任务次数”和“真正的修复次数”

当前演练记录的信息来自实际生成的 `experiment.json`，而非预先填的表格。

`coalesced_events` 表示合并的重复通知；`processed` 表示经历完整ACK的处理；`redrives` 则专门记录处理期间新增通知导致的再次入队。**这些量不互相等价。**

关键验收：同key重复不占额外槽，人工隔离与溢出阻断路由，Leader变更递增term，崩溃后修复不重复写入，真实物理KV最终归零。

In [23]:
if EXPERIMENT_DIR is not None:
    r=json.loads((EXPERIMENT_DIR/'experiment.json').read_text(encoding='utf-8'))
    for name in ('storm','safe_reconcile','danger','overflow','takeover'):
        print(name,':', {k:v for k,v in r[name].items() if k in {
            'events','unique_queue_items','coalesced','merged_events','route_blocked',
            'blocked_routing','full_resync_action','old_leader_fenced',
            'reclaimed_inflight','repaired_once','safe_after_full_resync'}})
    print('终态:',{'Leader':r['final']['lease_holder'],'term':r['final']['lease_term'],
        'WAL ack seq':r['final']['wal_ack_seq'],'KV物理页':r['final']['live_kv_blocks']})
    print('队列:',r['final']['queue'])

storm : {'coalesced': 47, 'events': 48, 'unique_queue_items': 1}
safe_reconcile : {'merged_events': 19}
danger : {'route_blocked': True}
overflow : {'blocked_routing': True, 'full_resync_action': 'ack', 'safe_after_full_resync': True}
takeover : {'old_leader_fenced': True, 'reclaimed_inflight': 1, 'repaired_once': True}
终态: {'Leader': 'controller_b', 'term': 2, 'WAL ack seq': 20, 'KV物理页': {'A': 0, 'B': 0}}
队列: {'accepted_events': 8, 'capacity': 3, 'coalesced_events': 66, 'depth': 0, 'full_resync_ready': False, 'inflight': 0, 'overflow': False, 'overflow_count': 1, 'parked': 0, 'pending': 0, 'processed': 8, 'reclaims': 1, 'redrives': 0, 'retries': 4}


## 19. 冷恢复检查：不能用同一个内存对象自证成功

独立地从磁盘加载模型、两份WAL、副本配置、控制Status、Lease和队列。验证：活动配置身份一致、Status观测代际匹配、没有未提交任务或溢出、真实KV对照通过。

下面先展示 Notebook 内的新对象恢复；另一个进程也可通过 `--reload-run` 进行相同校验。

In [24]:
if EXPERIMENT_DIR is not None:
    cold_report=reload_run(EXPERIMENT_DIR)
    print(json.dumps(cold_report,ensure_ascii=False,indent=2))

{
  "generation": 1,
  "config_sha": "b4b04155f3d18413e2fae5af622f6cb83e399f906ff1490dcef102ca6b6ca19b",
  "epoch": 1,
  "lease_holder": "controller_b",
  "lease_term": 2,
  "status_ready": true,
  "reconciliation_audit_head": "a7e42e608c564b7001e03f7aba53ebac2e7b26c4b1391cd20dad25daf6885e16",
  "config_audit_head": "9d144faa19f970b2829c438a3825fe37ecb7cb1d97343f5149da4446cb93135f",
  "workqueue_audit_head": "16b1f62b9fa2b04bf65d861e59686b86b30f42a7092a859aac4abc95f07e0991",
  "queue": {
    "capacity": 3,
    "depth": 0,
    "pending": 0,
    "inflight": 0,
    "parked": 0,
    "overflow": false,
    "overflow_count": 1,
    "full_resync_ready": false,
    "accepted_events": 8,
    "coalesced_events": 66,
    "processed": 8,
    "retries": 4,
    "reclaims": 1,
    "redrives": 0
  },
  "physical_KV_blocks": {
    "A": 0,
    "B": 0
  },
  "real_KV_full_logits_diff": 2.384185791015625e-07,
  "wal_ack_seq": 20,
  "not_real_distributed_consensus": true
}


## 20. 固定检查：不仅仅看“能运行”

测试覆盖：重入去重、FIFO、延迟重试不阻塞其他就绪资源、inflight的dirty通知、显式overflow及resync证据、Lease续约与旧票据围栏、term接管、危险漂移人工处理、崩溃后按实际状态恢复、状态或日志损坏的失败闭合，以及真实CPU KV/WAL。

这些检查只能说明本课程的串行状态机和有限模拟场景符合约定，**不是**对多进程锁、etcd、真实API、恶意篡改或GPU负载的认证。

In [25]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,value):
        if not bool(value):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,value,want):self.ok(name,value==want)
    def close(self,name,value,want,tol=2e-5):self.ok(name,abs(value-want)<=tol)
    def rejects(self,name,func):
        try:func()
        except (LessonError,ValueError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝却未拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    c=Checks()
    c.equal('课程唯一安全白名单',ReconcilePolicy().auto_paths,('telemetry_sample_percent',))
    c.equal('初始宽限2个tick',ReconcilePolicy().grace_ticks,2)
    c.equal('租约版本是本课',VERSION,'lesson85-workqueue-election-v1')
    c.ok('摘要规范确定性',digest({'x':1,'y':2})==digest({'y':2,'x':1}))
    c.ok('时间为逻辑tick非墙钟',SOURCES['leader_election'].startswith('https://kubernetes.io'))
    c.equal('空漂移健康',classify_drift([],ReconcilePolicy())['mode'],'healthy')
    c.equal('受控遥测漂移低风险',classify_drift([{'replica':'B','path':'telemetry_sample_percent',
             'status':'drift','severity':'controlled'}],ReconcilePolicy())['mode'],'auto_safe')
    c.equal('租户漂移不得自动修复',classify_drift([{'replica':'B','path':'allowed_tenants',
             'status':'drift','severity':'critical'}],ReconcilePolicy())['mode'],'manual')
    c.rejects('自动修复权限不能扩大',lambda:ReconcilePolicy(auto_paths=('route_canary_percent',)).validate())
    with tempfile.TemporaryDirectory() as temporary:
        root=Path(temporary)/'case'
        tok,models,cluster,store,loop,q,a,b=make_demo(root,8502,queue_capacity=3)
        c.equal('固定工作队列初始容量',q.stats()['capacity'],3)
        c.equal('初始队列为空',q.stats()['depth'],0)
        c.equal('初始无溢出',q.stats()['overflow'],False)
        c.equal('初始没有待清除resync',q.stats()['full_resync_ready'],False)
        c.ok('真实模型存在stable/canary',set(models)=={'stable','canary'})
        c.equal('初始2个物理副本',set(cluster.rep),{'A','B'})
        c.ok('两副本初始物理KV为零',all(cluster.rep[n].pool.audit()['live_blocks']==0 for n in ('A','B')))
        c.ok('初始配置相同',store.detect_drift()['healthy'])
        c.rejects('Follower禁止消费',lambda:b.process_one(0))
        c.rejects('Follower禁止路由',lambda:b.route(0,'alpha','without_leader'))
        old=a.elect(0,8)
        c.equal('首次租约term为1',old['term'],1)
        c.equal('首次租约持有者A',old['holder'],'controller_a')
        c.rejects('租期内另一个控制者不能选主',lambda:b.elect(1,8))
        status_before=loop.read_status();count_before=loop.events.verify()['count']
        read_only=b.observe()
        c.equal('Follower观察不改Status',loop.read_status(),status_before)
        c.equal('Follower观察不写审计',loop.events.verify()['count'],count_before)
        c.equal('Follower看到相同租约',read_only['lease'],a.ticket)
        c.rejects('租约名称不能含空格',lambda:loop.lease.acquire('bad name',1,8))
        c.rejects('续约TTL过小拒绝',lambda:renew_lease(loop.lease,a.ticket,0,1))
        c.rejects('续约不能时钟倒退',lambda:renew_lease(loop.lease,a.ticket,-1,8))
        rev=a.revision()
        c.equal('初始代际是1',rev['generation'],1)
        c.equal('初始cluster epoch是1',rev['epoch'],1)
        notices=[a.notify('cluster',0,'watch') for _ in range(12)]
        c.ok('重复事件首条入队',not notices[0]['coalesced'])
        c.ok('其余重复事件被合并',all(v['coalesced'] for v in notices[1:]))
        c.equal('12事件占一个槽',q.stats()['depth'],1)
        c.equal('合并数量明确计数',q.stats()['coalesced_events'],11)
        c.equal('合并仍只接受一新任务',q.stats()['accepted_events'],1)
        c.equal('同key多理由规范合并',a.notify('cluster',0,'periodic')['size'],1)
        c.equal('理由去重排序',q.read()['entries'][0]['reasons'],['periodic','watch'])
        c.rejects('未知资源key拒绝',lambda:q.enqueue('unknown',rev,0))
        c.rejects('非法资源revision拒绝',lambda:q.enqueue('A',{'generation':1,'sha':'x'},0))
        c.rejects('负逻辑tick拒绝',lambda:q.enqueue('A',rev,-1))
        claim=q.claim(a.ticket,0,loop.lease)
        c.equal('首次消费资源为cluster',claim['key'],'cluster')
        c.equal('消费后资源仍占队列',q.stats()['depth'],1)
        c.equal('inflight占一槽',q.stats()['inflight'],1)
        c.rejects('同资源处理中不会二次claim',lambda:q.ack({'holder':'other'},a.ticket,0,loop.lease))
        c.equal('第二个Get不能抢走inflight',q.claim(a.ticket,0,loop.lease),None)
        # 真正脏位测试：处理期间再次收到通知，ack后必须再入队。
        a.notify('cluster',0,'watch')
        c.ok('processing期间新的watch标记dirty',q.read()['entries'][0]['dirty'])
        result=q.ack(claim['claim'],a.ticket,0,loop.lease)
        c.equal('脏位ack不是最终移除',result['dirty_requeued'],True)
        c.equal('脏位任务仍待处理',q.stats()['depth'],1)
        c.equal('脏位重投计数',q.stats()['redrives'],1)
        c.equal('脏位回队列延后一个tick',q.claim(a.ticket,0,loop.lease),None)
        claim=q.claim(a.ticket,1,loop.lease)
        c.equal('脏位任务同key重取',claim['key'],'cluster')
        c.equal('重取后dirty清零',q.read()['entries'][0]['dirty'],False)
        q.ack(claim['claim'],a.ticket,1,loop.lease)
        c.equal('完整ack后队列空',q.stats()['depth'],0)
        c.equal('脏位引发恰好两次调谐槽消费',q.stats()['processed'],2)
        c.rejects('旧claim不能再次确认',lambda:q.ack(claim['claim'],a.ticket,1,loop.lease))
        # 事件只是提示，成功控制器以最新真实状态调谐。
        a.notify('cluster',2,'periodic')
        status=a.process_one(2)
        c.equal('初始化调谐Ready',status['reason'],'Ready')
        c.ok('真实模型权重一致性健康',store.detect_drift()['healthy'])
        kv=store.probe_kv()
        c.ok('真实增量KV和全前向一致',kv['max_cache_full_diff']<2e-5)
        req=a.route(2,'alpha','first','hello',4)
        c.equal('默认路由stable',req['version'],'stable')
        tokens=finish_request(store,req['rid'])
        c.ok('流式实际产生token事件',len(tokens)>0)
        c.equal('请求终态物理缓存清零',cluster.rep['A'].pool.audit()['live_blocks'],0)
        # 两级围栏：同term续约后的旧token失效；不同term接管后旧Leader无权操作。
        stale=copy.deepcopy(a.ticket)
        updated=a.heartbeat(3,8)
        c.equal('续约不更改term',updated['term'],stale['term'])
        c.ok('续约替换旧nonce',updated['token']!=stale['token'])
        c.rejects('续约后旧票据verify失败',lambda:loop.lease.verify(stale,3))
        c.rejects('续约后旧票据不能ack',lambda:q.claim(stale,3,loop.lease))
        c.rejects('同term下新Leader仍不得产生',lambda:b.elect(4,8))
        # FIFO，不让一个key的退避阻塞其他ready key。
        a.notify('A',4,'watch');a.notify('B',4,'watch')
        first=q.claim(a.ticket,4,loop.lease)
        c.equal('FIFO先A后B',first['key'],'A')
        retry=q.retry(first['claim'],a.ticket,4,loop.lease,delay=2,exponential=True)
        c.equal('第1次失败的最小等待2',retry['not_before'],6)
        second=q.claim(a.ticket,4,loop.lease)
        c.equal('延迟A不阻塞B',second['key'],'B')
        q.ack(second['claim'],a.ticket,4,loop.lease)
        c.equal('retry未到时claim应为空',q.claim(a.ticket,5,loop.lease),None)
        third=q.claim(a.ticket,6,loop.lease)
        c.equal('等待结束重新取A',third['key'],'A')
        retry2=q.retry(third['claim'],a.ticket,6,loop.lease,delay=1,cap=8,max_attempts=3)
        c.equal('第2次指数退避2',retry2['not_before'],8)
        fourth=q.claim(a.ticket,8,loop.lease)
        park=q.retry(fourth['claim'],a.ticket,8,loop.lease,delay=1,max_attempts=3)
        c.equal('达到重试上限需隔离',park['status'],'parked')
        c.equal('隔离任务不会自动重试',q.claim(a.ticket,8,loop.lease),None)
        c.equal('隔离任务占容量',q.stats()['parked'],1)
        q.release_parked('A',a.ticket,8,loop.lease)
        fifth=q.claim(a.ticket,8,loop.lease)
        c.equal('显式解隔离后可以消费',fifth['key'],'A')
        q.ack(fifth['claim'],a.ticket,8,loop.lease)
        c.equal('测试重试后队列为空',q.stats()['depth'],0)
        c.rejects('过去的逻辑tick不得入队',lambda:q.enqueue('A',rev,7))
        # 满队列必须持久overflow，清空不等于完成丢失事件的全量观察。
        for key in ('A','B','cluster'):a.notify(key,9,'watch')
        c.equal('填满3槽',q.stats()['depth'],3)
        c.rejects('第4个资源显式容量拒绝',lambda:a.notify('resync',9,'watch'))
        c.ok('overflow落盘并持久保留',q.stats()['overflow'])
        c.equal('记录容量拒绝次数',q.stats()['overflow_count'],1)
        c.rejects('overflow阻断新推理',lambda:a.route(9,'alpha','blocked','hello',4))
        for t in (9,10,11):
            item=q.claim(a.ticket,t,loop.lease)
            require(item is not None,'测试输入队列少于预期')
            q.ack(item['claim'],a.ticket,t,loop.lease)
        c.equal('清空队列仍是overflow',q.stats()['depth'],0)
        c.equal('清空队列不清overflow',q.stats()['overflow'],True)
        c.rejects('无完整resync不得解除overflow',lambda:q.clear_overflow_after_resync(a.ticket,11,loop.lease,store,loop))
        a.notify('resync',11,'full_resync')
        item=q.claim(a.ticket,11,loop.lease);q.ack(item['claim'],a.ticket,11,loop.lease)
        c.equal('完成resync任务后建立许可证据',q.stats()['full_resync_ready'],True)
        c.ok('只有真实Ready才能清除overflow',q.clear_overflow_after_resync(a.ticket,11,loop.lease,store,loop))
        c.equal('resync解锁后恢复正常',q.stats()['overflow'],False)
        c.equal('解锁消耗resync证据',q.stats()['full_resync_ready'],False)
        c.rejects('没有新的overflow和resync不能重复解锁',lambda:q.clear_overflow_after_resync(a.ticket,11,loop.lease,store,loop))
        # 租约A在续约t3时有效到11含端点；B必须t12取得下一term。
        c.rejects('到期边界不允许提前接管',lambda:b.elect(11,8))
        nxt=b.take_over(12,12)
        c.equal('接管term严格递增',nxt['term'],2)
        c.equal('队列没有遗留inflight',nxt['reclaimed'],0)
        c.rejects('旧Leader失去process权',lambda:a.process_one(12))
        c.rejects('旧Leader失去route权',lambda:a.route(12,'alpha','old'))
        c.rejects('旧Leader续约被拒',lambda:a.heartbeat(12,8))
        c.equal('新Leader也能只读观察',b.observe()['lease']['holder'],'controller_b')
        # 危险漂移：进入parked，不能被定时器重新投入正常队列。
        edit_replica_field(store,'B',allowed_tenants=['alpha'])
        b.notify('B',13,'watch')
        danger=b.process_one(13)
        c.equal('租户漂移必须进入人工状态',danger['reason'],'ManualReview')
        c.equal('危险队列标记park',danger['action'],'park')
        c.equal('权限漂移只占一个parked槽',q.stats()['parked'],1)
        c.rejects('危险漂移期间禁止路由',lambda:b.route(13,'alpha','unsafe','hello',4))
        approval=loop.manual_approval();manual=loop.manual_repair(b.ticket,14,approval)
        c.ok('真实人工夹具修复后ready',manual['status']['ready'])
        c.ok('危险字段修复后配置一致',store.detect_drift()['healthy'])
        q.release_parked('B',b.ticket,14,loop.lease)
        c.equal('人工解除park后可领任务',b.process_one(15)['action'],'ack')
        c.equal('人工修复后队列归零',q.stats()['depth'],0)
        # 崩溃after repair before queue ack；旧claim不会在新term直接被ack。
        edit_replica_field(store,'B',telemetry_sample_percent=76)
        for _ in range(8):b.notify('B',16,'watch')
        c.equal('相同漂移8次通知占一槽',q.stats()['depth'],1)
        c.equal('宽限第一tick',b.process_one(16)['reason'],'GracePeriod')
        c.equal('宽限第二tick',b.process_one(17)['reason'],'GracePeriod')
        config_count=store.audit.verify()['count']
        try:b.process_one(18,crash_after_repair=True)
        except SimulatedCrash:crashed=True
        else:crashed=False
        c.ok('确实发生修复后崩溃',crashed)
        c.equal('已提交修复恰好一次',store.audit.verify()['count'],config_count+1)
        c.equal('崩溃后的inflight仍持久',q.stats()['inflight'],1)
        c.ok('真实副本已经被修复',store.detect_drift()['healthy'])
        orphan=copy.deepcopy(q.read()['entries'][0]['claim'])
        c.rejects('同一term不能擅自回收inflight',lambda:q.recover_orphans(b.ticket,18,loop.lease))
        c.rejects('inflight旧票据不能造新ack',lambda:q.ack(orphan,a.ticket,18,loop.lease))
        # B在t12租期至24；A t25才能接管term3。
        c.rejects('B仍有效时A不能接管',lambda:a.elect(24,12))
        takeover=a.take_over(25,12)
        c.equal('再接管term为3',takeover['term'],3)
        c.equal('真实inflight被恢复',takeover['reclaimed'],1)
        c.equal('inflight重排进入pending',q.stats()['pending'],1)
        c.rejects('已失去term的B不能ack',lambda:q.ack(orphan,b.ticket,25,loop.lease))
        recovered=a.process_one(25)
        c.equal('新控制者恢复时读真实Ready',recovered['reason'],'Ready')
        c.equal('恢复重新观察后可以ack',recovered['action'],'ack')
        c.equal('修复没有重复落盘',store.audit.verify()['count'],config_count+1)
        c.equal('所有任务已结清',q.stats()['depth'],0)
        # 处理中的同代际watch必须不丢；不同代际在处理中必须升为overflow阻断。
        a.notify('A',26,'watch')
        claim=q.claim(a.ticket,26,loop.lease)
        a.notify('A',26,'watch')
        c.ok('再次通知使inflight变dirty',q.read()['entries'][0]['dirty'])
        ack=q.ack(claim['claim'],a.ticket,26,loop.lease)
        c.equal('ACK重新入队而不是丢失边沿',ack['dirty_requeued'],True)
        c.equal('dirty第二轮延迟至下个tick',q.claim(a.ticket,26,loop.lease),None)
        claim=q.claim(a.ticket,27,loop.lease);q.ack(claim['claim'],a.ticket,27,loop.lease)
        c.equal('重放后没有残留',q.stats()['depth'],0)
        a.notify('A',28,'watch')
        claim=q.claim(a.ticket,28,loop.lease)
        higher={'generation':rev['generation']+1,'sha':'0'*64,'epoch':rev['epoch']}
        c.rejects('inflight出现更高代际必须要求resync',lambda:q.enqueue('A',higher,28,'watch'))
        c.ok('inflight冲突设置overflow',q.stats()['overflow'])
        q.ack(claim['claim'],a.ticket,28,loop.lease)
        c.rejects('仅ack不足以解除代际溢出',lambda:q.clear_overflow_after_resync(a.ticket,28,loop.lease,store,loop))
        a.notify('resync',29,'full_resync')
        r=q.claim(a.ticket,29,loop.lease);q.ack(r['claim'],a.ticket,29,loop.lease)
        c.ok('新resync恢复路由安全',q.clear_overflow_after_resync(a.ticket,29,loop.lease,store,loop))
        # 冷启动：模型权重、WAL、Status和队列必须全部通过磁盘检查。
        restored=make_demo(root,load_existing=True)
        q2=restored[5]
        c.equal('冷启动队列深度相同',q2.stats()['depth'],q.stats()['depth'])
        c.equal('冷启动overflow状态相同',q2.stats()['overflow'],q.stats()['overflow'])
        c.equal('队列日志完整性通过',q.events.verify()['count'],q2.events.verify()['count'])
        c.ok('实际序列增量KV仍正确',store.probe_kv()['max_cache_full_diff']<2e-5)
        c.equal('全部物理KV归零',{n:cluster.rep[n].pool.audit()['live_blocks'] for n in ('A','B')},{'A':0,'B':0})
        # 数据损坏与篡改：不因只靠seal而自动修复，返回明确错误。
        before_data=q.path.read_bytes()
        raw=json.loads(before_data);raw['data']['clock']+=1
        q.path.write_text(canonical(raw)+'\n',encoding='utf-8')
        c.rejects('队列内容改动但seal不变会拒绝',lambda:q.read())
        q.path.write_bytes(before_data)
        c.ok('恢复原始队列文件后可读',q.read()['clock']>=29)
        before_audit=q.events.path.read_bytes()
        corrupted=before_audit.replace(b'queue_ack',b'queue_bad',1)
        require(corrupted!=before_audit,'测试审计日志未包含ack')
        q.events.path.write_bytes(corrupted)
        c.rejects('变更日志串改被发现',lambda:q.events.verify())
        q.events.path.write_bytes(before_audit)
        c.ok('日志还原后可以验证',q.events.verify()['count']>0)
        c.rejects('无效队列容量拒绝',lambda:DurableWorkQueue(root/'bad_queue',capacity=0))
        c.rejects('队列不能不经load_existing覆盖',lambda:DurableWorkQueue(root/'workqueue'))
        c.rejects('配置退避不能取bool',lambda:ReconcilePolicy(grace_ticks=True).validate())
        c.rejects('同资源错误哈希不能合并',lambda:q.enqueue('A',{'generation':1,'sha':'x'*64,'epoch':1},30))
    return {'passed':len(c.names),'names':c.names,
            'scope':'single-process lease/workqueue with actual CPU Paged KV and durable WAL',
            'not_tested':['multi-process race','etcd/CAS','Kubernetes API','real IAM','GPU latency']}

In [26]:
checks_report=run_checks()
print('固定检查通过:',checks_report['passed'],'项')
print('范围:',checks_report['scope'])
if EXPERIMENT_DIR is not None:
    write_json(EXPERIMENT_DIR/'checks.json',checks_report)

固定检查通过: 145 项
范围: single-process lease/workqueue with actual CPU Paged KV and durable WAL


## 21. 练习：先自己实现，再运行参考答案

三道练习依次检测事件合并、重试计算、租约围栏。每一道都设置了隐藏的失败情况；默认检查开关为False，不替你完成练习。检查函数代码可以展开阅读。

In [27]:
def check_exercise_a(fn:Callable):
    events=[{'key':'A','generation':1},{'key':'B','generation':1},
            {'key':'A','generation':1},{'key':'A','generation':2},
            {'key':'B','generation':1}]
    origin=copy.deepcopy(events)
    got=fn(events)
    require(got=={'keys':['A','B'],'latest_generation':{'A':2,'B':1},'coalesced':3},
            '练习A需保序、同key合并并保留最高代际')
    require(events==origin,'练习A不可原地修改输入')
    try:fn([{'key':'A','generation':2},{'key':'A','generation':1}])
    except ValueError:pass
    else:raise LessonError('练习A必须拒绝倒退代际')
    print('练习A通过：保序去重、代际推进和反例。')
    return True


def check_exercise_b(fn:Callable):
    require([fn(n,base=1,cap=8,grace=False) for n in range(1,7)]==[1,2,4,8,8,8],
            '练习B指数退避或上限不正确')
    require([fn(n,base=2,cap=8,grace=True) for n in range(1,5)]==[1,1,1,1],
            'GracePeriod不能继承故障指数退避')
    for bad in [0,-1,True]:
        try:fn(bad,base=1,cap=8,grace=False)
        except ValueError:pass
        else:raise LessonError('练习B不应接收错误attempt')
    print('练习B通过：指数退避、宽限重排与非法输入。')
    return True


def check_exercise_c(fn:Callable):
    old={'holder':'A','term':1,'cluster_epoch':1,'issued_tick':0,'expires_tick':8,'token':'old'}
    now={**old,'issued_tick':4,'expires_tick':12,'token':'new'}
    require(fn(now,now,12,1),'练习C过期边界包含端点')
    require(not fn(old,now,4,1),'练习C续约旧票据需要失效')
    require(not fn(now,now,13,1),'练习C过期后不能通过')
    require(not fn(now,now,8,2),'练习C旧cluster epoch应拒绝')
    require(not fn({**now,'holder':'B'},now,8,1),'练习C非持有者应拒绝')
    try:fn(now,now,True,1)
    except ValueError:pass
    else:raise LessonError('练习C不能接受bool作为tick')
    print('练习C通过：租约续约围栏、时间端点与cluster epoch。')
    return True

### 练习 A：保序合并 key，保留最新代际

输入例如：`A(1),B(1),A(1),A(2),B(1)`。输出保序的 `[A,B]`、最终代际 `{A:2,B:1}` 和合并数3。禁止悄悄接受同key从2倒退到1；不能修改原输入列表。

In [28]:
def fold_events_ex(events):
    # TODO: 只按稳定key合并，不把watch内容当作可执行配置。
    raise NotImplementedError('请独立完成练习A')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    check_exercise_a(fold_events_ex)

### 练习 B：指数重试与观察宽限

实现 `delay_ex(attempt,base=1,cap=8,grace=False)`。普通故障是 `base·2^(attempt−1)`，受 `cap` 限制；宽限检查固定一个tick。要拒绝 bool 伪装的整数、零和负attempt。

In [29]:
def delay_ex(attempt,base=1,cap=8,grace=False):
    # TODO: 观察宽限和失败重试不能混淆。
    raise NotImplementedError('请独立完成练习B')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    check_exercise_b(delay_ex)

### 练习 C：租约门控与续约票据

纯函数 `lease_gate_ex(ticket,current,tick,epoch)` 返回 bool：要求票据和当前权威记录字段内容一致，epoch相同，tick处在包含端点的有效期内。拒绝bool作为tick；旧票据即使holder/term相同也无权写。

In [30]:
def lease_gate_ex(ticket,current,tick,epoch):
    # TODO: 必须比较完整票据，不只比较holder/term。
    raise NotImplementedError('请独立完成练习C')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    check_exercise_c(lease_gate_ex)

## 22. 完整参考答案

下面分别命名为 `answer_a/b/c`，不会覆盖上方留给你的函数。完成后再阅读，并检查拒绝分支。生产环境还需要授权、统一时钟、共享事务、指标采样和外部隔离机制。

In [31]:
def answer_a(events:list[dict]) -> dict:
    order=[];last={};coalesced=0
    for item in events:
        require(type(item) is dict and set(item)=={'key','generation'},'事件字段错误')
        key=item['key'];gen=item['generation']
        require(type(key) is str and bool(key) and type(gen) is int and gen>=1,'事件非法')
        if key in last:
            require(gen>=last[key],'代际不能倒退')
            coalesced+=1
        else:order.append(key)
        last[key]=gen
    return {'keys':order,'latest_generation':last,'coalesced':coalesced}


def answer_b(attempt:int,base:int=1,cap:int=8,grace:bool=False) -> int:
    require(type(attempt) is int and attempt>=1 and type(base) is int and base>=1
            and type(cap) is int and cap>=base and type(grace) is bool,'退避输入非法')
    if grace:return 1
    return min(cap,base*2**(attempt-1))


def answer_c(ticket:dict,current:dict,tick:int,epoch:int) -> bool:
    require(type(tick) is int and tick>=0 and type(epoch) is int and epoch>=1,
            'tick与epoch必须是非负/正整数且非bool')
    require(type(ticket) is dict and type(current) is dict,'lease需为字典')
    fields=('holder','term','cluster_epoch','issued_tick','expires_tick','token')
    require(all(k in ticket and k in current for k in fields),'lease字段缺失')
    return (ticket==current and ticket['cluster_epoch']==epoch
            and ticket['issued_tick']<=tick<=ticket['expires_tick'])

In [32]:
check_exercise_a(answer_a)
check_exercise_b(answer_b)
check_exercise_c(answer_c)

练习A通过：保序去重、代际推进和反例。
练习B通过：指数退避、宽限重排与非法输入。
练习C通过：租约续约围栏、时间端点与cluster epoch。


True

## 23. 独立脚本与本课限制

配套 `.py` 内含同样的模型、队列、恢复、演练、145项固定检查以及参考答案。Notebook 的 `main()` 定义也在下一单元；Notebook 不自动执行 `if __name__ == '__main__'`，避免重复触发命令行入口。

主要输出文件：`manifest.json`、`experiment.json`、`reload.json`、`checks.json`；状态位于 `cluster/`、`config/`、`controller/`、`workqueue/`，都可以独立审计。工作队列的 `queue_events.jsonl` 是哈希链日志，但局部事件日志与状态文件**不是跨文件原子提交**；若在任意两次写入间宕机，需要外部事务/一致性修复。这份代码刻意不冒充完整数据库。

**其他不能承诺的能力：** Kubernetes Lease API、真正多进程选举、防恶意身份伪造、磁盘跨主机一致性、真实节点故障注入、真实GPU延迟和生产级性能。逻辑tick是刻意设计用于确定性测试的模拟时钟。

In [33]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    grp=parser.add_mutually_exclusive_group()
    grp.add_argument('--checks-only',action='store_true')
    grp.add_argument('--math-only',action='store_true')
    grp.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_85_outputs'))
    parser.add_argument('--seed',type=int,default=8501)
    opts=parser.parse_args(argv)
    torch.set_num_threads(1)
    if opts.reload_run:
        print(json.dumps(reload_run(opts.reload_run),ensure_ascii=False,indent=2))
        return 0
    if opts.math_only:
        print(json.dumps({'backoff':[answer_b(i) for i in range(1,7)],
                'grace':[answer_b(i,grace=True) for i in range(1,5)],
                'not_real_consensus':True,'logical_ticks_not_seconds':True,
                'source':SOURCES['workqueue']},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['passed']}项；真实CPU KV/持久WAL、单进程双控制者教学模拟。")
    if opts.checks_only:return 0
    folder=run_experiment(opts.output_dir,opts.seed)
    write_json(folder/'checks.json',checks)
    print('独立冷恢复：',json.dumps(reload_run(folder),ensure_ascii=False))
    return 0

## 24. 资料与下一课

- [Kubernetes Controller 模式](https://kubernetes.io/docs/concepts/architecture/controller/)
- [Kubernetes Coordinated Leader Election](https://kubernetes.io/docs/concepts/cluster-administration/coordinated-leader-election/)
- [client-go WorkQueue 类型、去重与限速](https://pkg.go.dev/k8s.io/client-go/util/workqueue)
- [client-go WorkQueue 使用示例](https://github.com/kubernetes/client-go/blob/master/examples/workqueue/main.go)
- [PyTorch 可复现性限制](https://docs.pytorch.org/docs/stable/notes/randomness.html)

下课建议：**第86课——控制器并发安全、原子事务与故障注入线性化**。在本课的串行单进程模拟之外，讨论多进程下哪些操作必须由外部原子存储保护，为什么文件替换不能自动提供分布式共识。